In [1]:
# Cell 1: Imports
import os
import sys
import csv
import json
import time
import random
import hashlib
import platform
import struct
import subprocess
import datetime
import itertools
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

In [2]:
# Cell 2: Configuration
AUG0 = {
    "reflect_pad": 4,
    "crop_size": 32,
    "hflip": True,
    "mix": "none",
    "cutout_size": 16,
    "mixup_alpha": 0.4,
    "cutmix_alpha": 1.0,
    "area_min": 1.0,
    "aspect_max": 1.0,
    "rot_deg": 0.0,
    "cj_brightness": 0.0,
    "cj_contrast": 0.0,
    "cj_saturation": 0.0,
}
TRAIN0 = {
    "momentum": 0.9,
    "eval_batch_size": 256,
    "print_every": 10,
    "flip_avg": True,
    "optimizer": "sgd",
    "label_smoothing": 0.0,
    "warmup_epochs": 0,
    "ema_decay": 0.0,
    "dropout": 0.0,
}
CONFIG = {
    "seeds": {"split": 42, "train": (0, 1, 2)},
    "data": {
        "train_images": "dataset/train-images-idx3-ubyte",
        "train_labels": "dataset/train-labels-idx1-ubyte",
        "labels_file": "dataset/labels.txt",
        "num_train": 1979,
        "image_shape": (32, 32, 3),
        "num_classes": 10,
        "expected_class_counts": (217, 216, 195, 220, 204, 204, 178, 180, 174, 191),
    },
    "paths": {
        "results_dir": "results",
        "folds_file": "results/folds.json",
        "metrics_csv": "results/metrics.csv",
        "final_checkpoint": "results/final_model.pt",
    },
    "dedup": {
        "rms_threshold": 5.0,
        "audit_thresholds": (1.0, 2.0, 3.0, 5.0, 10.0, 20.0),
        "pair_sample_size": 24,
    },
    "cv": {"n_folds": 5},
    "augmentation": dict(AUG0),
    "baseline": {
        "augmentation": dict(AUG0),
        "train": dict(TRAIN0),
        "optimizer": "sgd",
    },
    "train": dict(TRAIN0),
    "expected_params": {
        "smallcnn": 24458,
        "vgg_16_32_64": 72954,
        "resnet_16_32_64": 78042,
        "resnet_32_64_128": 308650,
        "resnet_harmonic": 308650,
        "resnet_depth2": 696618,
        "resnet_up64": 308650,
        "resnet_48_96_192": 691834,
    },
    "queue": [
        {
            "models": ("resnet_32_64_128",),
            "lrs": (0.1,),
            "wds": (2e-3,),
            "epochs": 240,
            "batch_size": 32,
            "seeds": (0, 1, 2),
        },
    ],
    "defaults": {
        "model": "resnet_32_64_128",
        "lr": 0.1,
        "wd": 2e-3,
        "epochs": 240,
        "batch_size": 32,
        "seeds": (0, 1, 2),
    },
    "experiments": {
        "REG": {"seeds": (0,), "suffix": "_reg", "regression": True},
        "E1": {"aug": {"area_min": 0.6, "aspect_max": 1.25, "rot_deg": 15.0}},
        "LR05": {"lr": 0.05},
        "LR02": {"lr": 0.2},
        "LS01": {"train": {"label_smoothing": 0.1}},
        "EMA": {"train": {"ema_decay": 0.999}},
        "WARM": {"train": {"warmup_epochs": 5}},
        "DROP": {"train": {"dropout": 0.2}},
        "E2": {"aug": {"cj_brightness": 0.3, "cj_contrast": 0.3, "cj_saturation": 0.3}},
        "MIX02": {"aug": {"mix": "mixup", "mixup_alpha": 0.2}},
        "MIX10": {"aug": {"mix": "mixup", "mixup_alpha": 1.0}},
        "BS64": {"batch_size": 64},
        "WD1": {"wd": 1e-3},
        "ADAMW_A": {"lr": 2e-3, "wd": 0.05, "train": {"optimizer": "adamw"}},
        "ADAMW_B": {"lr": 1e-3, "wd": 0.05, "train": {"optimizer": "adamw"}},
        "EP480": {"epochs": 480},
        "DROP_EP480": {"epochs": 480, "train": {"dropout": 0.2}},
        "DROP_EP480_MIX": {"epochs": 480, "aug": {"mix": "mixup"}, "train": {"dropout": 0.2}},
        "DROP_EP480_CUTOUT": {"epochs": 480, "aug": {"mix": "cutout"}, "train": {"dropout": 0.2}},
        "DROP_EP480_CUTMIX": {"epochs": 480, "aug": {"mix": "cutmix"}, "train": {"dropout": 0.2}},
    },
    "final": None,
    "final_check": {"parity_max_mismatch_frac": 0.005},
    "runtime": {
        "deterministic": True,
        "cudnn_benchmark": False,
        "cudnn_allow_tf32": False,
        "cublas_workspace_config": ":4096:8",
        "nvidia_smi_timeout_s": 10,
    },
}

print(json.dumps(CONFIG, indent=2))

{
  "seeds": {
    "split": 42,
    "train": [
      0,
      1,
      2
    ]
  },
  "data": {
    "train_images": "dataset/train-images-idx3-ubyte",
    "train_labels": "dataset/train-labels-idx1-ubyte",
    "labels_file": "dataset/labels.txt",
    "num_train": 1979,
    "image_shape": [
      32,
      32,
      3
    ],
    "num_classes": 10,
    "expected_class_counts": [
      217,
      216,
      195,
      220,
      204,
      204,
      178,
      180,
      174,
      191
    ]
  },
  "paths": {
    "results_dir": "results",
    "folds_file": "results/folds.json",
    "metrics_csv": "results/metrics.csv",
    "final_checkpoint": "results/final_model.pt"
  },
  "dedup": {
    "rms_threshold": 5.0,
    "audit_thresholds": [
      1.0,
      2.0,
      3.0,
      5.0,
      10.0,
      20.0
    ],
    "pair_sample_size": 24
  },
  "cv": {
    "n_folds": 5
  },
  "augmentation": {
    "reflect_pad": 4,
    "crop_size": 32,
    "hflip": true,
    "mix": "none",
    "cutout_size"

In [3]:
# Cell 3: Environment record and seeding
RT = CONFIG["runtime"]

if torch.cuda.is_initialized() and os.environ.get("CUBLAS_WORKSPACE_CONFIG") != RT["cublas_workspace_config"]:
    raise RuntimeError("CUDA was initialised before CUBLAS_WORKSPACE_CONFIG was set; restart the kernel and run Cells 1-3 first")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = RT["cublas_workspace_config"]

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available; refusing to fall back to CPU. Check the GPU driver, the PyTorch CUDA build and WSL2 GPU access")

DEVICE = torch.device("cuda:0")
capability = torch.cuda.get_device_capability(0)
arch_needed = f"sm_{capability[0]}{capability[1]}"
arch_list = torch.cuda.get_arch_list()
if arch_needed not in arch_list:
    raise RuntimeError(f"Installed PyTorch build lists {arch_list} but the GPU needs {arch_needed}; install a build with CUDA 12.8 or newer")

torch.use_deterministic_algorithms(RT["deterministic"])
torch.backends.cudnn.deterministic = RT["deterministic"]
torch.backends.cudnn.benchmark = RT["cudnn_benchmark"]
torch.backends.cudnn.allow_tf32 = RT["cudnn_allow_tf32"]


def smoke_test():
    x = torch.randn(4, 3, 8, 8, device=DEVICE, requires_grad=True)
    conv = torch.nn.Conv2d(3, 4, 3, padding=1).to(DEVICE)
    out = (conv(x).mean(dim=(2, 3)) @ torch.ones(4, 2, device=DEVICE)).sum()
    out.backward()
    torch.cuda.synchronize()
    return bool(torch.isfinite(x.grad).all().item())


if not smoke_test():
    raise RuntimeError("CUDA conv/matmul forward-backward smoke test produced non-finite gradients")

try:
    driver = subprocess.run(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=RT["nvidia_smi_timeout_s"], check=True,
    ).stdout.strip()
except (OSError, subprocess.SubprocessError) as exc:
    driver = f"unavailable ({type(exc).__name__})"

props = torch.cuda.get_device_properties(0)
ENV_RECORD = {
    "timestamp": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "venv": sys.prefix,
    "numpy": np.__version__,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "driver": driver,
    "gpu": props.name,
    "gpu_capability": f"{capability[0]}.{capability[1]}",
    "gpu_memory_gb": round(props.total_memory / 1024**3, 2),
    "arch_list": arch_list,
    "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
    "float32_matmul_precision": torch.get_float32_matmul_precision(),
    "cublas_workspace_config": os.environ["CUBLAS_WORKSPACE_CONFIG"],
    "seeds": CONFIG["seeds"],
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seeds"]["split"])

print(json.dumps(ENV_RECORD, indent=2, default=str))
print(f"Cell 3 done: device={DEVICE}, seed={CONFIG['seeds']['split']}")

{
  "timestamp": "2026-10-24T14:40:37+00:00",
  "python": "3.13.17",
  "platform": "Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.43",
  "venv": "/home/rushan/DogImagePrediction/.venv313",
  "numpy": "2.5.4",
  "torch": "2.13.0+cu132",
  "torch_cuda": "13.2",
  "cudnn": 92000,
  "driver": "616.64",
  "gpu": "NVIDIA GeForce RTX 5070",
  "gpu_capability": "12.0",
  "gpu_memory_gb": 11.91,
  "arch_list": [
    "sm_75",
    "sm_80",
    "sm_86",
    "sm_90",
    "sm_100",
    "sm_120"
  ],
  "deterministic_algorithms": true,
  "cudnn_deterministic": true,
  "cudnn_benchmark": false,
  "cudnn_allow_tf32": false,
  "float32_matmul_precision": "highest",
  "cublas_workspace_config": ":4096:8",
  "seeds": {
    "split": 42,
    "train": [
      0,
      1,
      2
    ]
  }
}
Cell 3 done: device=cuda:0, seed=42


In [4]:
# Cell 4: Data loading and integrity checks
D = CONFIG["data"]
ROWS, COLS, CH = D["image_shape"]


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def load_dog_mnist(images_path, labels_path):
    with open(labels_path, "rb") as f:
        magic, n_labels = struct.unpack(">II", f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    if magic != 2049 or labels.size != n_labels:
        raise ValueError(f"Bad label file: magic={magic}, header count={n_labels}, bytes={labels.size}")
    with open(images_path, "rb") as f:
        magic, n, rows, cols = struct.unpack(">IIII", f.read(16))
        pixels = np.frombuffer(f.read(), dtype=np.uint8)
    if magic != 2051 or n != n_labels or (rows, cols) != (ROWS, COLS) or pixels.size != n * ROWS * COLS * CH:
        raise ValueError(f"Bad image file: magic={magic}, n={n}, rows={rows}, cols={cols}, bytes={pixels.size}, labels={n_labels}")
    return pixels.reshape(n, ROWS, COLS, CH).copy(), labels.astype(np.int64)


X_TRAIN, Y_TRAIN = load_dog_mnist(D["train_images"], D["train_labels"])

if X_TRAIN.shape != (D["num_train"], ROWS, COLS, CH) or X_TRAIN.dtype != np.uint8:
    raise ValueError(f"Unexpected image array: shape={X_TRAIN.shape}, dtype={X_TRAIN.dtype}")
if Y_TRAIN.min() < 0 or Y_TRAIN.max() >= D["num_classes"]:
    raise ValueError(f"Label out of range: min={Y_TRAIN.min()}, max={Y_TRAIN.max()}")

class_counts = np.bincount(Y_TRAIN, minlength=D["num_classes"])
if class_counts.tolist() != list(D["expected_class_counts"]):
    raise ValueError(f"Class counts differ from README: got {class_counts.tolist()}, expected {list(D['expected_class_counts'])}")

image_std = X_TRAIN.reshape(len(X_TRAIN), -1).astype(np.float32).std(axis=1)
if image_std.min() <= 0:
    raise ValueError(f"Constant image found at indices {np.flatnonzero(image_std <= 0).tolist()}")

by_bytes = {}
for i in range(len(X_TRAIN)):
    by_bytes.setdefault(X_TRAIN[i].tobytes(), []).append(i)
EXACT_DUP_GROUPS = [g for g in by_bytes.values() if len(g) > 1]
dup_label_conflicts = [g for g in EXACT_DUP_GROUPS if len(set(Y_TRAIN[g].tolist())) > 1]
if dup_label_conflicts:
    raise ValueError(f"Exact duplicates with conflicting labels: {dup_label_conflicts}")

with open(D["labels_file"], encoding="utf-8") as f:
    LABELS_RAW = [line.rstrip("\n") for line in f if line.strip()]

DATA_RECORD = {
    "train_images_sha256": sha256_file(D["train_images"]),
    "train_labels_sha256": sha256_file(D["train_labels"]),
    "num_images": int(X_TRAIN.shape[0]),
    "class_counts": class_counts.tolist(),
    "pixel_min": int(X_TRAIN.min()),
    "pixel_max": int(X_TRAIN.max()),
    "min_image_std": round(float(image_std.min()), 3),
    "exact_duplicate_groups": len(EXACT_DUP_GROUPS),
    "exact_duplicate_images": sum(len(g) for g in EXACT_DUP_GROUPS),
}

print(json.dumps(DATA_RECORD, indent=2))
print("exact duplicate groups:", EXACT_DUP_GROUPS)
print("labels file:", LABELS_RAW)
print(f"Cell 4 done: X_TRAIN={X_TRAIN.shape} {X_TRAIN.dtype}, Y_TRAIN={Y_TRAIN.shape} {Y_TRAIN.dtype}")

{
  "train_images_sha256": "d6f6798d9679c31813f8d9001c167e1072c4efc155ea54bf404fe4000aa9e477",
  "train_labels_sha256": "795de4627e931d999be38a9c0d5df92c0689f1c78f2f287ef8d22618df8fa713",
  "num_images": 1979,
  "class_counts": [
    217,
    216,
    195,
    220,
    204,
    204,
    178,
    180,
    174,
    191
  ],
  "pixel_min": 0,
  "pixel_max": 255,
  "min_image_std": 16.186,
  "exact_duplicate_groups": 2,
  "exact_duplicate_images": 4
}
exact duplicate groups: [[30, 1321], [402, 1978]]
labels file: ['0\tBedlington_terrier', '1\tBoston_bull', '2\tEnglish_setter', '3\tItalian_greyhound', '4\tLabrador_retriever', '5\tNorfolk_terrier', '6\tWalker_hound', '7\tboxer', '8\tgolden_retriever', '9\twire-haired_fox_terrier']
Cell 4 done: X_TRAIN=(1979, 32, 32, 3) uint8, Y_TRAIN=(1979,) int64


In [5]:
# Cell 5: Near-duplicate audit
DD = CONFIG["dedup"]
N_IMG = len(X_TRAIN)
FLAT = X_TRAIN.reshape(N_IMG, -1).astype(np.float64)
FLAT_FLIP = X_TRAIN[:, :, ::-1, :].reshape(N_IMG, -1).astype(np.float64)
DIM = FLAT.shape[1]
SQ = (FLAT ** 2).sum(axis=1)


def rms_matrix(a, b):
    d2 = SQ[:, None] + SQ[None, :] - 2.0 * (a @ b.T)
    return np.sqrt(np.clip(d2, 0.0, None) / DIM)


RMS_DIRECT = rms_matrix(FLAT, FLAT)
RMS_FLIP = rms_matrix(FLAT, FLAT_FLIP)
np.fill_diagonal(RMS_DIRECT, np.inf)
np.fill_diagonal(RMS_FLIP, np.inf)
RMS_PAIR = np.minimum(RMS_DIRECT, RMS_FLIP)
del FLAT, FLAT_FLIP


def cluster_ids(thr):
    parent = np.arange(N_IMG)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    ii, jj = np.nonzero(np.triu(RMS_PAIR <= thr, k=1))
    for a, b in zip(ii.tolist(), jj.tolist()):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    roots = np.array([find(i) for i in range(N_IMG)])
    return np.unique(roots, return_inverse=True)[1].reshape(-1)


def summarize(thr):
    ids = cluster_ids(thr)
    sizes = np.bincount(ids)
    multi = np.flatnonzero(sizes > 1)
    conflicts = sum(len(set(Y_TRAIN[ids == c].tolist())) > 1 for c in multi)
    return ids, {
        "thr": float(thr),
        "direct_pairs": int(np.triu(RMS_DIRECT <= thr, k=1).sum()),
        "flip_only_pairs": int(np.triu((RMS_FLIP <= thr) & (RMS_DIRECT > thr), k=1).sum()),
        "images_in_groups": int(sizes[multi].sum()),
        "clusters": int(len(sizes)),
        "max_cluster": int(sizes.max()),
        "label_conflicts": int(conflicts),
    }


print(f"{'thr':>6} {'direct':>7} {'flipOnly':>8} {'imgsInGrp':>9} {'clusters':>8} {'maxSize':>7} {'conflicts':>9}")
for t in DD["audit_thresholds"]:
    r = summarize(t)[1]
    print(f"{r['thr']:6.1f} {r['direct_pairs']:7d} {r['flip_only_pairs']:8d} {r['images_in_groups']:9d} {r['clusters']:8d} {r['max_cluster']:7d} {r['label_conflicts']:9d}")

CLUSTER_IDS, NEAR_DUP_RECORD = summarize(DD["rms_threshold"])
CLUSTER_SIZES = np.bincount(CLUSTER_IDS)
if NEAR_DUP_RECORD["label_conflicts"]:
    raise ValueError(f"Label conflicts inside near-duplicate clusters at threshold {DD['rms_threshold']}")
sizes_u, sizes_c = np.unique(CLUSTER_SIZES, return_counts=True)
NEAR_DUP_RECORD["cluster_size_histogram"] = {int(s): int(c) for s, c in zip(sizes_u, sizes_c)}

thr = DD["rms_threshold"]
iu, ju = np.nonzero(np.triu(RMS_PAIR <= 2 * thr, k=1))
vals = RMS_PAIR[iu, ju]
order = np.argsort(vals, kind="stable")
iu, ju, vals = iu[order], ju[order], vals[order]
half = DD["pair_sample_size"] // 2
selected = {}
if len(vals):
    for s in np.unique(np.linspace(0, len(vals) - 1, half).astype(int)):
        selected[int(s)] = "spread"
    for s in np.argsort(np.abs(vals - thr), kind="stable")[:half]:
        selected[int(s)] = "near-threshold"

print(f"\nchosen threshold {thr}; pairs with RMS <= {2 * thr}: {len(vals)}")
print(f"{'i':>5} {'j':>5} {'rmsDirect':>9} {'rmsFlip':>8} {'yi':>3} {'yj':>3} {'ci':>5} {'cj':>5} {'same':>5}  tag")
for s in sorted(selected):
    i, j = int(iu[s]), int(ju[s])
    print(f"{i:5d} {j:5d} {RMS_DIRECT[i, j]:9.3f} {RMS_FLIP[i, j]:8.3f} {Y_TRAIN[i]:3d} {Y_TRAIN[j]:3d} {CLUSTER_IDS[i]:5d} {CLUSTER_IDS[j]:5d} {str(CLUSTER_IDS[i] == CLUSTER_IDS[j]):>5}  {selected[s]}")

print("\n" + json.dumps(NEAR_DUP_RECORD, indent=2))
print(f"Cell 5 done: {NEAR_DUP_RECORD['clusters']} clusters at threshold {thr}")

   thr  direct flipOnly imgsInGrp clusters maxSize conflicts
   1.0     481        0       959     1499       3         0
   2.0     516        0      1021     1467       4         0
   3.0     518        0      1022     1466       4         0
   5.0     518        0      1022     1466       4         0
  10.0     518        0      1022     1466       4         0
  20.0     524        0      1023     1464       4         0

chosen threshold 5.0; pairs with RMS <= 10.0: 518
    i     j rmsDirect  rmsFlip  yi  yj    ci    cj  same  tag
   30  1321     0.000   83.866   6   6    30    30  True  spread
  819   997     0.602   32.309   7   7   729   729  True  spread
  665  1968     0.637   86.198   1   1   599   599  True  spread
  194  1877     0.654   51.891   8   8   191   191  True  spread
  588  1424     0.672   54.569   0   0   539   539  True  spread
  568   858     0.687   41.812   4   4   522   522  True  spread
 1160  1707     0.707   64.511   7   7   978   978  True  spread
  188

In [6]:
# Cell 6: Group-aware stratified folds
K = CONFIG["cv"]["n_folds"]
NC = CONFIG["data"]["num_classes"]
Path(CONFIG["paths"]["results_dir"]).mkdir(parents=True, exist_ok=True)


def build_folds(ids, labels, k, seed):
    n_clusters = int(ids.max()) + 1
    cl_counts = np.zeros((n_clusters, NC), dtype=np.int64)
    np.add.at(cl_counts, (ids, labels), 1)
    rng = np.random.default_rng(seed)
    order = np.lexsort((rng.random(n_clusters), -cl_counts.sum(axis=1)))
    target = cl_counts.sum(axis=0) / k
    fold_counts = np.zeros((k, NC))
    cluster_fold = np.full(n_clusters, -1, dtype=np.int64)
    for c in order:
        cost = ((fold_counts + cl_counts[c] - target) ** 2 - (fold_counts - target) ** 2).sum(axis=1)
        f = int(np.argmin(cost))
        cluster_fold[c] = f
        fold_counts[f] += cl_counts[c]
    return cluster_fold[ids]


FOLD_ID = build_folds(CLUSTER_IDS, Y_TRAIN, K, CONFIG["seeds"]["split"])

n_clusters = int(CLUSTER_IDS.max()) + 1
fold_min = np.full(n_clusters, K)
fold_max = np.full(n_clusters, -1)
np.minimum.at(fold_min, CLUSTER_IDS, FOLD_ID)
np.maximum.at(fold_max, CLUSTER_IDS, FOLD_ID)
assert (fold_min == fold_max).all(), "a near-duplicate cluster crosses folds"
assert all(len(set(FOLD_ID[g].tolist())) == 1 for g in EXACT_DUP_GROUPS), "an exact-duplicate group crosses folds"
assert set(FOLD_ID.tolist()) == set(range(K)), "empty fold"

FOLD_CLASS_COUNTS = np.stack([np.bincount(Y_TRAIN[FOLD_ID == f], minlength=NC) for f in range(K)])
assert (FOLD_CLASS_COUNTS > 0).all(), "a class is missing from a fold"
assert FOLD_CLASS_COUNTS.sum() == len(Y_TRAIN)

FOLD_SPLITS = [(np.flatnonzero(FOLD_ID != f), np.flatnonzero(FOLD_ID == f)) for f in range(K)]

FOLDS_PAYLOAD = {
    "split_seed": CONFIG["seeds"]["split"],
    "rms_threshold": CONFIG["dedup"]["rms_threshold"],
    "n_folds": K,
    "train_images_sha256": DATA_RECORD["train_images_sha256"],
    "fold_id": FOLD_ID.tolist(),
    "cluster_id": CLUSTER_IDS.tolist(),
}
folds_path = Path(CONFIG["paths"]["folds_file"])
if folds_path.exists():
    if json.loads(folds_path.read_text()) != FOLDS_PAYLOAD:
        raise RuntimeError(f"{folds_path} differs from the folds built now; seed, threshold or data changed. Delete it deliberately if intended")
    print(f"folds match saved file {folds_path}")
else:
    folds_path.write_text(json.dumps(FOLDS_PAYLOAD))
    print(f"folds saved to {folds_path}")

print("fold sizes:", [len(va) for _, va in FOLD_SPLITS])
print("per-fold class counts (rows=folds):")
print(FOLD_CLASS_COUNTS)
print("max |fold class count - target|:", float(np.abs(FOLD_CLASS_COUNTS - np.bincount(Y_TRAIN, minlength=NC) / K).max()))
print("Cell 6 done")

folds match saved file results/folds.json
fold sizes: [399, 397, 396, 395, 392]
per-fold class counts (rows=folds):
[[44 44 39 44 41 41 36 36 35 39]
 [44 43 39 44 41 41 36 36 35 38]
 [43 43 39 44 41 41 36 36 35 38]
 [43 43 39 44 41 41 35 36 35 38]
 [43 43 39 44 40 40 35 36 34 38]]
max |fold class count - target|: 0.7999999999999972
Cell 6 done


In [7]:
# Cell 7: Data tensors, normalization and augmentation
AUG = CONFIG["augmentation"]
PAD, SIZE = AUG["reflect_pad"], AUG["crop_size"]
MIX = AUG["mix"]
CUTOUT = AUG["cutout_size"]
if MIX not in ("none", "cutout", "cutmix", "mixup", "random"):
    raise ValueError(f"Unknown augmentation mix mode: {MIX}")
if not 1 <= CUTOUT <= SIZE:
    raise ValueError(f"cutout_size {CUTOUT} must be between 1 and {SIZE}")
if not 0.0 < AUG["area_min"] <= 1.0:
    raise ValueError(f"area_min {AUG['area_min']} must be in (0, 1]")
if AUG["aspect_max"] < 1.0:
    raise ValueError(f"aspect_max {AUG['aspect_max']} must be >= 1")
if AUG["rot_deg"] < 0.0:
    raise ValueError(f"rot_deg {AUG['rot_deg']} must be >= 0")
for _k in ("cj_brightness", "cj_contrast", "cj_saturation"):
    if not 0.0 <= AUG[_k] < 1.0:
        raise ValueError(f"{_k} {AUG[_k]} must be in [0, 1)")
X_ALL = torch.from_numpy(X_TRAIN).permute(0, 3, 1, 2).contiguous().to(DEVICE)
Y_ALL = torch.from_numpy(Y_TRAIN).to(DEVICE)


def norm_stats(idx):
    x = X_ALL[torch.as_tensor(idx, device=DEVICE)].float() / 255.0
    return x.mean(dim=(0, 2, 3)), x.std(dim=(0, 2, 3), correction=0)


def normalize(idx, mean, std):
    x = X_ALL[torch.as_tensor(idx, device=DEVICE)].float() / 255.0
    return (x - mean[None, :, None, None]) / std[None, :, None, None]


def affine_on():
    return AUG["area_min"] < 1.0 or AUG["aspect_max"] > 1.0 or AUG["rot_deg"] > 0.0


def cj_on():
    return AUG["cj_brightness"] > 0.0 or AUG["cj_contrast"] > 0.0 or AUG["cj_saturation"] > 0.0


def epoch_plan(n, gen):
    perm = torch.randperm(n, generator=gen)
    oy = torch.randint(0, 2 * PAD + 1, (n,), generator=gen)
    ox = torch.randint(0, 2 * PAD + 1, (n,), generator=gen)
    flip = (torch.rand(n, generator=gen) < 0.5) & bool(AUG["hflip"])
    extra = torch.rand(n, 3, generator=gen).to(DEVICE) if affine_on() else None
    cj = torch.rand(n, 3, generator=gen).to(DEVICE) if cj_on() else None
    return perm.to(DEVICE), oy.to(DEVICE), ox.to(DEVICE), flip.to(DEVICE), extra, cj


@torch.no_grad()
def affine_jitter(x, extra):
    u0, u1, u2 = extra[:, 0], extra[:, 1], extra[:, 2]
    side = torch.sqrt(1.0 - u0 * (1.0 - AUG["area_min"]))
    ratio = torch.exp((2.0 * u1 - 1.0) * float(np.log(AUG["aspect_max"])))
    ax = (side * torch.sqrt(ratio)).clamp(max=1.0)
    ay = (side / torch.sqrt(ratio)).clamp(max=1.0)
    ang = (2.0 * u2 - 1.0) * AUG["rot_deg"] * float(np.pi) / 180.0
    c, s = torch.cos(ang), torch.sin(ang)
    zero = torch.zeros_like(c)
    theta = torch.stack([
        torch.stack([c * ax, -s * ay, zero], dim=1),
        torch.stack([s * ax, c * ay, zero], dim=1),
    ], dim=1)
    grid = F.affine_grid(theta, tuple(x.shape), align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)


@torch.no_grad()
def colour_jitter(x, u, mean, std):
    m, s = mean[None, :, None, None], std[None, :, None, None]
    img = x * s + m
    b = (1.0 + (2.0 * u[:, 0] - 1.0) * AUG["cj_brightness"])[:, None, None, None]
    c = (1.0 + (2.0 * u[:, 1] - 1.0) * AUG["cj_contrast"])[:, None, None, None]
    sat = (1.0 + (2.0 * u[:, 2] - 1.0) * AUG["cj_saturation"])[:, None, None, None]
    img = img * b
    mu = img.mean(dim=(1, 2, 3), keepdim=True)
    img = (img - mu) * c + mu
    gray = 0.299 * img[:, 0:1] + 0.587 * img[:, 1:2] + 0.114 * img[:, 2:3]
    img = (img - gray) * sat + gray
    img = img.clamp(0.0, 1.0)
    return (img - m) / s


@torch.no_grad()
def augment(xb, oy, ox, flip, extra=None):
    b, c = xb.shape[0], xb.shape[1]
    padded = F.pad(xb, (PAD, PAD, PAD, PAD), mode="reflect")
    ar = torch.arange(SIZE, device=xb.device)
    rows = (oy[:, None] + ar[None, :])[:, None, :, None]
    cols = (ox[:, None] + ar[None, :])[:, None, None, :]
    bi = torch.arange(b, device=xb.device)[:, None, None, None]
    ci = torch.arange(c, device=xb.device)[None, :, None, None]
    out = padded[bi, ci, rows, cols]
    out = torch.where(flip[:, None, None, None], out.flip(3), out)
    if extra is None:
        return out
    return affine_jitter(out, extra)


def make_mix_rng(seed):
    return np.random.default_rng(seed + 1000)


def pick_method(rng):
    if MIX == "random":
        return ("cutout", "mixup", "cutmix")[int(rng.integers(3))]
    return MIX


def box_mask(y0, x0, h, w):
    ar = torch.arange(SIZE, device=DEVICE)
    rm = (ar[None, :] >= y0[:, None]) & (ar[None, :] < (y0 + h)[:, None])
    cm = (ar[None, :] >= x0[:, None]) & (ar[None, :] < (x0 + w)[:, None])
    return (rm[:, :, None] & cm[:, None, :])[:, None]


@torch.no_grad()
def mix_batch(xb, yb, rng, method):
    if method == "none":
        return xb, None, 1.0
    b = xb.shape[0]
    if method == "cutout":
        y0 = torch.as_tensor(rng.integers(SIZE, size=b) - CUTOUT // 2, device=DEVICE)
        x0 = torch.as_tensor(rng.integers(SIZE, size=b) - CUTOUT // 2, device=DEVICE)
        return xb.masked_fill(box_mask(y0, x0, CUTOUT, CUTOUT), 0.0), None, 1.0
    partner_x, partner_y = xb.flip(0), yb.flip(0)
    if method == "mixup":
        lam = float(rng.beta(AUG["mixup_alpha"], AUG["mixup_alpha"]))
        return lam * xb + (1.0 - lam) * partner_x, partner_y, lam
    lam0 = float(rng.beta(AUG["cutmix_alpha"], AUG["cutmix_alpha"]))
    side = int(SIZE * float(np.sqrt(1.0 - lam0)))
    cy, cx = int(rng.integers(SIZE)), int(rng.integers(SIZE))
    ys, xs = cy - side // 2, cx - side // 2
    y1, y2 = max(ys, 0), min(ys + side, SIZE)
    x1, x2 = max(xs, 0), min(xs + side, SIZE)
    out = xb.clone()
    out[:, :, y1:y2, x1:x2] = partner_x[:, :, y1:y2, x1:x2]
    lam = 1.0 - (y2 - y1) * (x2 - x1) / (SIZE * SIZE)
    return out, partner_y, lam


_t = torch.arange(2 * 3 * SIZE * SIZE, dtype=torch.float32, device=DEVICE).reshape(2, 3, SIZE, SIZE)
_mid = torch.full((2,), PAD, device=DEVICE)
_zero = torch.zeros(2, dtype=torch.long, device=DEVICE)
_no, _yes = torch.zeros(2, dtype=torch.bool, device=DEVICE), torch.ones(2, dtype=torch.bool, device=DEVICE)
assert torch.equal(augment(_t, _mid, _mid, _no), _t)
assert torch.equal(augment(_t, _mid, _mid, _yes), _t.flip(3))
assert torch.equal(augment(_t, _zero, _zero, _no)[:, :, PAD:, PAD:], _t[:, :, :SIZE - PAD, :SIZE - PAD])

_ai = torch.randn(4, 3, SIZE, SIZE, device=DEVICE)
_ident = torch.tensor([[0.0, 0.5, 0.5]] * 4, device=DEVICE)
assert torch.allclose(affine_jitter(_ai, _ident), _ai, atol=1e-5)
assert torch.allclose(augment(_ai, _mid[:1].expand(4), _mid[:1].expand(4), _no[:1].expand(4), _ident), _ai, atol=1e-5)
if affine_on():
    _ex = torch.rand(4, 3, generator=torch.Generator().manual_seed(0)).to(DEVICE)
    _aj = affine_jitter(_ai, _ex)
    assert _aj.shape == _ai.shape and torch.isfinite(_aj).all()
    assert not torch.allclose(_aj, _ai, atol=1e-3)

_cm = torch.full((3,), 0.5, device=DEVICE)
_cs = torch.full((3,), 0.25, device=DEVICE)
_cx = (torch.rand(4, 3, SIZE, SIZE, device=DEVICE) - 0.5) / 0.25
_half = torch.full((4, 3), 0.5, device=DEVICE)
assert torch.allclose(colour_jitter(_cx, _half, _cm, _cs), _cx, atol=1e-4)
if cj_on():
    _cy = colour_jitter(_cx, torch.rand(4, 3, device=DEVICE), _cm, _cs)
    _raw = _cy * 0.25 + 0.5
    assert torch.isfinite(_cy).all() and _raw.min() >= -1e-5 and _raw.max() <= 1.0 + 1e-5
    assert not torch.allclose(_cy, _cx, atol=1e-3)

_rng = make_mix_rng(0)
_xb = torch.randn(8, 3, SIZE, SIZE, device=DEVICE)
_yb = torch.arange(8, device=DEVICE)
_x, _y2, _lam = mix_batch(_xb, _yb, _rng, "none")
assert _x is _xb and _y2 is None and _lam == 1.0
_x, _y2, _lam = mix_batch(_xb, _yb, _rng, "cutout")
assert _y2 is None and _lam == 1.0 and (_x == 0).any()
assert not ((_x != _xb) & (_x != 0)).any()
assert ((_x != _xb).float().mean(dim=(1, 2, 3)) <= CUTOUT * CUTOUT / (SIZE * SIZE) + 1e-6).all()
_x, _y2, _lam = mix_batch(_xb, _yb, _rng, "mixup")
assert 0.0 <= _lam <= 1.0 and torch.equal(_y2, _yb.flip(0))
assert torch.allclose(_x, _lam * _xb + (1.0 - _lam) * _xb.flip(0), atol=1e-6)
_x, _y2, _lam = mix_batch(_xb, _yb, _rng, "cutmix")
assert 0.0 <= _lam <= 1.0 and torch.equal(_y2, _yb.flip(0))
assert abs((_x != _xb).any(dim=1).float().mean().item() - (1.0 - _lam)) < 1e-5
assert make_mix_rng(0).integers(1000) == make_mix_rng(0).integers(1000)

print(f"Cell 7 done: X_ALL={tuple(X_ALL.shape)} {X_ALL.dtype} on {X_ALL.device}; mix={MIX}; affine={'on' if affine_on() else 'off'}; colour={'on' if cj_on() else 'off'}")

Cell 7 done: X_ALL=(1979, 3, 32, 32) torch.uint8 on cuda:0; mix=none; affine=off; colour=off


In [8]:
# Cell 8: Model definitions and parameter-count check
import math

def conv_bn_relu(cin, cout, bias):
    return [nn.Conv2d(cin, cout, 3, padding=1, bias=bias), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]


class Plain(nn.Module):
    def __init__(self, widths, convs_per_stage, bias, num_classes):
        super().__init__()
        layers, cin = [], 3
        for s, w in enumerate(widths):
            for _ in range(convs_per_stage):
                layers += conv_bn_relu(cin, w, bias)
                cin = w
            if s < len(widths) - 1:
                layers.append(nn.MaxPool2d(2))
        self.features = nn.Sequential(*layers)
        self.fc = nn.Linear(cin, num_classes)

    def forward(self, x):
        return self.fc(self.features(x).mean(dim=(2, 3)))


class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.short = None
        if stride != 1 or cin != cout:
            self.short = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return F.relu(y + (x if self.short is None else self.short(x)))


class ResNet(nn.Module):
    def __init__(self, widths, num_classes):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        blocks, cin = [], widths[0]
        for s, w in enumerate(widths):
            blocks.append(Block(cin, w, 1 if s == 0 else 2))
            cin = w
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(cin, num_classes)

    def forward(self, x):
        return self.fc(self.blocks(self.stem(x)).mean(dim=(2, 3)))


# ── Candidate #6: Deeper ResNet (2 blocks per stage, 13 conv layers) ──
class ResNetDepth2(nn.Module):
    def __init__(self, widths, num_classes):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        blocks, cin = [], widths[0]
        for s, w in enumerate(widths):
            blocks.append(Block(cin, w, 1 if s == 0 else 2))
            blocks.append(Block(w, w, 1))
            cin = w
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(cin, num_classes)

    def forward(self, x):
        return self.fc(self.blocks(self.stem(x)).mean(dim=(2, 3)))


# ── Candidate #9: Upsampled Stem ResNet (anti-aliased subpixel sampling) ──
class ResNetUp64(nn.Module):
    def __init__(self, widths, num_classes):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, widths[0], 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(widths[0]),
            nn.ReLU(inplace=True)
        )
        blocks, cin = [], widths[0]
        for s, w in enumerate(widths):
            blocks.append(Block(cin, w, 1 if s == 0 else 2))
            cin = w
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(cin, num_classes)

    def forward(self, x):
        x_up = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        return self.fc(self.blocks(self.stem(x_up)).mean(dim=(2, 3)))


# ── Candidate #8: Harmonic ResNet (fixed 2D-DCT basis convolutions) ──
def get_dct_filter_bank(K=3):
    filters = []
    for u in range(K):
        for v in range(K):
            filt = torch.zeros(K, K)
            for x in range(K):
                for y in range(K):
                    filt[x, y] = math.cos(math.pi * (x + 0.5) * u / K) * math.cos(math.pi * (y + 0.5) * v / K)
            norm = torch.norm(filt)
            if norm > 0:
                filt = filt / norm
            filters.append(filt)
    basis = torch.stack(filters).unsqueeze(1)
    return basis


class HarmonicConv2d(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.cin = cin
        self.cout = cout
        self.stride = stride
        basis = get_dct_filter_bank(3)
        full_basis = basis.repeat(cin, 1, 1, 1)
        self.register_buffer("basis", full_basis)
        self.pointwise = nn.Conv2d(cin * 9, cout, kernel_size=1, stride=1, bias=False)

    def forward(self, x):
        feat = F.conv2d(x, self.basis, stride=self.stride, padding=1, groups=self.cin)
        return self.pointwise(feat)


class HarmonicBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = HarmonicConv2d(cin, cout, stride)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = HarmonicConv2d(cout, cout, 1)
        self.bn2 = nn.BatchNorm2d(cout)
        self.short = None
        if stride != 1 or cin != cout:
            self.short = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return F.relu(y + (x if self.short is None else self.short(x)))


class ResNetHarmonic(nn.Module):
    def __init__(self, widths, num_classes):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        blocks, cin = [], widths[0]
        for s, w in enumerate(widths):
            blocks.append(HarmonicBlock(cin, w, 1 if s == 0 else 2))
            cin = w
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(cin, num_classes)

    def forward(self, x):
        return self.fc(self.blocks(self.stem(x)).mean(dim=(2, 3)))


def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)


MODEL_BUILDERS = {
    "smallcnn": lambda nc: Plain((16, 32, 64), 1, True, nc),
    "vgg_16_32_64": lambda nc: Plain((16, 32, 64), 2, False, nc),
    "resnet_16_32_64": lambda nc: ResNet((16, 32, 64), nc),
    "resnet_32_64_128": lambda nc: ResNet((32, 64, 128), nc),
    "resnet_depth2": lambda nc: ResNetDepth2((32, 64, 128), nc),
    "resnet_48_96_192": lambda nc: ResNet((48, 96, 192), nc),
    "resnet_up64": lambda nc: ResNetUp64((32, 64, 128), nc),
    "resnet_harmonic": lambda nc: ResNetHarmonic((32, 64, 128), nc),
}


def build_model(key):
    m = MODEL_BUILDERS[key](CONFIG["data"]["num_classes"])
    m.apply(init_weights)
    return m.to(DEVICE)


def count_params(m):
    return sum(p.numel() for p in m.parameters())


for key in MODEL_BUILDERS:
    m = build_model(key).eval()
    with torch.no_grad():
        out_shape = tuple(m(torch.zeros(2, 3, SIZE, SIZE, device=DEVICE)).shape)
    got, want = count_params(m), CONFIG["expected_params"][key]
    print(f"{key:18s} params={got:8d} expected={want:8d} out={out_shape}")
    assert got == want, f"{key}: parameter count {got} != expected {want}"
    assert out_shape == (2, CONFIG["data"]["num_classes"])
    del m

print("Cell 8 done")


smallcnn           params=   24458 expected=   24458 out=(2, 10)
vgg_16_32_64       params=   72954 expected=   72954 out=(2, 10)
resnet_16_32_64    params=   78042 expected=   78042 out=(2, 10)
resnet_32_64_128   params=  308650 expected=  308650 out=(2, 10)
resnet_depth2      params=  696618 expected=  696618 out=(2, 10)
resnet_48_96_192   params=  691834 expected=  691834 out=(2, 10)
resnet_up64        params=  308650 expected=  308650 out=(2, 10)
resnet_harmonic    params=  308650 expected=  308650 out=(2, 10)
Cell 8 done


In [9]:
# Cell 9: Training and validation
import copy
import math

T = CONFIG["train"]


@torch.no_grad()
def predict_logits(model, x, batch_size):
    model.eval()
    do_flip = T.get("flip_avg", True)
    preds = []
    for start in range(0, x.shape[0], batch_size):
        end = min(start + batch_size, x.shape[0])
        xb = x[start:end]
        logits = model(xb)
        if do_flip:
            logits = (logits + model(xb.flip(3))) / 2
        preds.append(logits)
    return torch.cat(preds)


def make_optimizer(model, lr, wd):
    params = list(model.parameters())
    groups = [
        {"params": [p for p in params if p.ndim > 1], "weight_decay": wd},
        {"params": [p for p in params if p.ndim <= 1], "weight_decay": 0.0},
    ]
    if T["optimizer"] == "sgd":
        return torch.optim.SGD(groups, lr=lr, momentum=T["momentum"])
    if T["optimizer"] == "adamw":
        return torch.optim.AdamW(groups, lr=lr, betas=(0.9, 0.999))
    raise ValueError(f"Unknown optimizer: {T['optimizer']}")


def make_scheduler(opt, epochs, steps):
    w = int(T["warmup_epochs"]) * steps
    total = epochs * steps
    if w == 0:
        return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total)

    def factor(i):
        if i < w:
            return (i + 1) / w
        return 0.5 * (1.0 + math.cos(math.pi * (i - w) / max(1, total - w)))

    return torch.optim.lr_scheduler.LambdaLR(opt, factor)


def attach_dropout(model):
    p = float(T["dropout"])
    if p > 0.0:
        model.fc.register_forward_pre_hook(lambda mod, inp: (F.dropout(inp[0], p, mod.training),))
    return model


@torch.no_grad()
def ema_update(ema, model, decay):
    for e, m in zip(ema.state_dict().values(), model.state_dict().values()):
        if e.dtype.is_floating_point:
            e.mul_(decay).add_(m, alpha=1.0 - decay)
        else:
            e.copy_(m)


def train_one(model_key, lr, wd, epochs, batch_size, seed, train_idx, val_idx, prefix=""):
    set_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    mix_rng = make_mix_rng(seed)
    model = attach_dropout(build_model(model_key))
    ema = None
    if T["ema_decay"] > 0.0:
        ema = copy.deepcopy(model)
        for p in ema.parameters():
            p.requires_grad_(False)
    eval_model = model if ema is None else ema
    ls = float(T["label_smoothing"])
    mean, std = norm_stats(train_idx)
    x_tr = normalize(train_idx, mean, std)
    y_tr = Y_ALL[torch.as_tensor(train_idx, device=DEVICE)]
    if val_idx is not None:
        x_va = normalize(val_idx, mean, std)
        y_va = Y_ALL[torch.as_tensor(val_idx, device=DEVICE)]
    n = x_tr.shape[0]
    steps = n // batch_size
    opt = make_optimizer(model, lr, wd)
    sched = make_scheduler(opt, epochs, steps)
    history, logits, t0, step_count = [], None, time.time(), 0
    bar = tqdm(range(epochs), desc=prefix.strip() or "train", leave=False, dynamic_ncols=True)
    for epoch in bar:
        model.train()
        perm, oy, ox, flip, extra, cj = epoch_plan(n, gen)
        loss_sum = torch.zeros((), device=DEVICE)
        for s in range(steps):
            b = perm[s * batch_size:(s + 1) * batch_size]
            xb = augment(x_tr[b], oy[b], ox[b], flip[b], None if extra is None else extra[b])
            if cj is not None:
                xb = colour_jitter(xb, cj[b], mean, std)
            yb = y_tr[b]
            xb, y2, lam = mix_batch(xb, yb, mix_rng, pick_method(mix_rng))
            out = model(xb)
            if y2 is None:
                loss = F.cross_entropy(out, yb, label_smoothing=ls)
            else:
                loss = lam * F.cross_entropy(out, yb, label_smoothing=ls) + (1.0 - lam) * F.cross_entropy(out, y2, label_smoothing=ls)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            if ema is not None:
                ema_update(ema, model, min(float(T["ema_decay"]), (1.0 + step_count) / (10.0 + step_count)))
            step_count += 1
            loss_sum += loss.detach()
        train_loss = loss_sum.item() / steps
        if not np.isfinite(train_loss):
            raise FloatingPointError(f"{prefix}non-finite training loss at epoch {epoch + 1}")
        val_loss = val_acc = float("nan")
        if val_idx is not None:
            logits = predict_logits(eval_model, x_va, T["eval_batch_size"])
            val_loss = F.cross_entropy(logits, y_va).item()
            val_acc = (logits.argmax(dim=1) == y_va).float().mean().item()
        history.append((epoch + 1, train_loss, val_loss, val_acc))
        bar.set_postfix(loss=f"{train_loss:.3f}", val_acc=f"{val_acc:.3f}")
        if (epoch + 1) % T["print_every"] == 0 or epoch + 1 == epochs:
            tqdm.write(f"{prefix}ep {epoch + 1:3d}/{epochs} train_loss {train_loss:.4f} val_loss {val_loss:.4f} val_acc {val_acc:.4f} lr {sched.get_last_lr()[0]:.5f}")
    bar.close()
    return {
        "model": eval_model,
        "mean": mean,
        "std": std,
        "history": history,
        "val_logits": None if logits is None else logits.float().cpu().numpy(),
        "params": count_params(eval_model),
        "seconds": time.time() - t0,
    }


print("Cell 9 done")

Cell 9 done


In [10]:
# Cell 10: Experiment runner
RUN_NAMES = ["DROP_EP480", "DROP_EP480_MIX", "DROP_EP480_CUTOUT", "DROP_EP480_CUTMIX"]

RES = Path(CONFIG["paths"]["results_dir"])
METRIC_FIELDS = ["run_id", "model", "lr", "wd", "epochs", "batch_size", "seed", "fold", "n_train", "n_val", "val_acc", "val_loss", "train_loss", "params", "seconds"]
BASE_AUG = CONFIG["baseline"]["augmentation"]
BASE_TRAIN = CONFIG["baseline"]["train"]
TAG_TRAIN_KEYS = ("optimizer", "label_smoothing", "warmup_epochs", "ema_decay", "dropout")
EXPERIMENTS = CONFIG["experiments"]
DEFAULTS = CONFIG["defaults"]
SUFFIX = ""
CURRENT = ""


def recipe_tag():
    parts = []
    for k in sorted(set(AUG) | set(BASE_AUG)):
        if AUG.get(k) != BASE_AUG.get(k):
            parts.append(f"{k}{AUG.get(k)}")
    for k in TAG_TRAIN_KEYS:
        if T.get(k) != BASE_TRAIN.get(k):
            parts.append(str(T.get(k)) if k == "optimizer" else f"{k}{T.get(k)}")
    return "".join("_" + p.replace(" ", "").replace("/", "") for p in parts) + SUFFIX


def run_id_for(model_key, lr, wd, epochs, batch_size):
    return f"{model_key}_lr{lr:g}_wd{wd:g}_ep{epochs}_bs{batch_size}{recipe_tag()}"


def check_cfg():
    if AUG["mix"] not in ("none", "cutout", "cutmix", "mixup", "random"):
        raise ValueError(f"Unknown mix mode: {AUG['mix']}")
    if not 0.0 < AUG["area_min"] <= 1.0 or AUG["aspect_max"] < 1.0 or AUG["rot_deg"] < 0.0:
        raise ValueError("invalid affine settings")
    for k in ("cj_brightness", "cj_contrast", "cj_saturation"):
        if not 0.0 <= AUG[k] < 1.0:
            raise ValueError(f"{k} must be in [0, 1)")
    if T["optimizer"] not in ("sgd", "adamw"):
        raise ValueError(f"Unknown optimizer: {T['optimizer']}")
    if not 0.0 <= T["label_smoothing"] < 1.0 or not 0.0 <= T["ema_decay"] < 1.0 or not 0.0 <= T["dropout"] < 1.0:
        raise ValueError("label_smoothing, ema_decay and dropout must be in [0, 1)")
    if int(T["warmup_epochs"]) != T["warmup_epochs"] or T["warmup_epochs"] < 0:
        raise ValueError("warmup_epochs must be a non-negative integer")


def apply_experiment(name):
    global SUFFIX, CURRENT
    exp = EXPERIMENTS[name]
    AUG.clear()
    AUG.update(BASE_AUG)
    AUG.update(exp.get("aug", {}))
    T.clear()
    T.update(BASE_TRAIN)
    T.update(exp.get("train", {}))
    SUFFIX = exp.get("suffix", "")
    CURRENT = name
    globals()["MIX"] = AUG["mix"]
    check_cfg()


def append_rows(rows):
    p = Path(CONFIG["paths"]["metrics_csv"])
    new = not p.exists()
    with open(p, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=METRIC_FIELDS)
        if new:
            w.writeheader()
        w.writerows(rows)


def run_cv(model_key, lr, wd, epochs, batch_size, seed):
    rid = run_id_for(model_key, lr, wd, epochs, batch_size)
    oof_path = RES / f"oof_{rid}_s{seed}.npz"
    if oof_path.exists():
        tqdm.write(f"SKIP {rid} seed {seed}: {oof_path.name} exists")
        return None
    oof_logits = np.zeros((len(Y_TRAIN), NC), dtype=np.float32)
    rows = []
    for f, (tr, va) in tqdm(list(enumerate(FOLD_SPLITS)), desc=f"folds {rid} s{seed}", leave=False, dynamic_ncols=True):
        out = train_one(model_key, lr, wd, epochs, batch_size, seed, tr, va, prefix=f"[{rid} s{seed} f{f}] ")
        oof_logits[va] = out["val_logits"]
        last = out["history"][-1]
        rows.append({
            "run_id": rid, "model": model_key, "lr": lr, "wd": wd, "epochs": epochs, "batch_size": batch_size,
            "seed": seed, "fold": f, "n_train": len(tr), "n_val": len(va), "val_acc": round(last[3], 6),
            "val_loss": round(last[2], 6), "train_loss": round(last[1], 6), "params": out["params"],
            "seconds": round(out["seconds"], 1),
        })
        del out
    preds = oof_logits.argmax(axis=1)
    run_config = {
        "experiment": CURRENT, "model": model_key, "lr": lr, "wd": wd, "epochs": epochs, "batch_size": batch_size,
        "seed": seed, "momentum": T["momentum"], "optimizer": T["optimizer"], "flip_avg": bool(T["flip_avg"]),
        "train": dict(T), "augmentation": dict(AUG), "rms_threshold": DD["rms_threshold"],
        "split_seed": CONFIG["seeds"]["split"], "n_folds": K,
    }
    np.savez(oof_path, logits=oof_logits, preds=preds, labels=Y_TRAIN, fold_id=FOLD_ID, config=json.dumps(run_config))
    append_rows(rows)
    pooled = float((preds == Y_TRAIN).mean())
    secs = [r["seconds"] for r in rows]
    tqdm.write(f"DONE {rid} seed {seed}: pooled OOF acc {pooled:.4f}; {sum(secs):.0f} s total, {np.mean(secs):.1f} s/fold")
    return pooled


def load_oof(rid, seeds):
    out = []
    for s in seeds:
        p = RES / f"oof_{rid}_s{s}.npz"
        if not p.exists():
            return None
        with np.load(p) as z:
            out.append(z["logits"].astype(np.float64))
    return out


def regression_check(model_key, lr, wd, epochs, batch_size, seed):
    new_id = run_id_for(model_key, lr, wd, epochs, batch_size)
    old_id = new_id[:-len(SUFFIX)] if SUFFIX else new_id
    pn, po = RES / f"oof_{new_id}_s{seed}.npz", RES / f"oof_{old_id}_s{seed}.npz"
    if not po.exists():
        tqdm.write(f"REGRESSION: reference file {po.name} missing, check not possible")
        return
    with np.load(pn) as a, np.load(po) as b:
        diff = float(np.abs(a["logits"].astype(np.float64) - b["logits"].astype(np.float64)).max())
        flips = int((a["preds"] != b["preds"]).sum())
        pa, pb = float((a["preds"] == a["labels"]).mean()), float((b["preds"] == b["labels"]).mean())
    tqdm.write(f"REGRESSION seed {seed}: new {pa:.4f} vs reference {pb:.4f}; max |logit diff| {diff:.3e}; changed predictions {flips}")
    if abs(pa - pb) > 0.003:
        raise RuntimeError("regression check failed: baseline path no longer reproduces the reference; stop and report")
    tqdm.write("REGRESSION: pass" if diff < 1e-4 else "REGRESSION: pooled accuracy within 0.003 but logits differ; report this")


def summarize(name, spec):
    new_id = run_id_for(spec["model"], spec["lr"], spec["wd"], spec["epochs"], spec["batch_size"])
    ref_id = f"{spec['model']}_lr{DEFAULTS['lr']:g}_wd{DEFAULTS['wd']:g}_ep{DEFAULTS['epochs']}_bs{DEFAULTS['batch_size']}" + ("" if AUG["mix"] == "none" else f"_mix{AUG['mix']}")
    new, ref = load_oof(new_id, spec["seeds"]), load_oof(ref_id, spec["seeds"])
    if new is None or ref is None:
        tqdm.write(f"SUMMARY {name}: skipped, OOF files missing (new {new_id}, reference {ref_id})")
        return
    hits = lambda lg: (lg.argmax(axis=1) == Y_TRAIN).astype(np.float64)
    sn, sr = np.mean([hits(a) for a in new], axis=0), np.mean([hits(a) for a in ref], axis=0)
    en, er = hits(np.mean(new, axis=0)), hits(np.mean(ref, axis=0))
    d, de = sn - sr, en - er
    delta = d.mean() * 100
    verdict = "CLEAR GAIN" if delta >= 2.0 else ("clearly worse" if delta <= -2.0 else "inconclusive")
    tqdm.write(f"SUMMARY {name} vs {ref_id} ({len(spec['seeds'])} seeds, flip-averaged)")
    tqdm.write(f"  single mean: new {sn.mean():.4f} ref {sr.mean():.4f} delta {delta:+.2f} points, paired SE {d.std(ddof=1) / np.sqrt(len(d)) * 100:.2f}  -> {verdict}")
    tqdm.write(f"  ensemble:    new {en.mean():.4f} ref {er.mean():.4f} delta {de.mean() * 100:+.2f} points, paired SE {de.std(ddof=1) / np.sqrt(len(de)) * 100:.2f}")
    tqdm.write("  rule: clear gain needs >= +2.0 single-mean points; between -2.0 and +2.0 is inconclusive")


unknown = [n for n in RUN_NAMES if n not in EXPERIMENTS]
if unknown:
    raise KeyError(f"Unknown experiment names: {unknown}")

SESSION_FILE = RES / f"session_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
SESSION_FILE.write_text(json.dumps({"env": ENV_RECORD, "data": DATA_RECORD, "near_duplicates": NEAR_DUP_RECORD, "run_names": RUN_NAMES, "config": CONFIG}, indent=2, default=str))

for name in RUN_NAMES:
    exp = EXPERIMENTS[name]
    apply_experiment(name)
    spec = {**DEFAULTS, **{k: v for k, v in exp.items() if k in DEFAULTS}}
    if spec["model"] not in MODEL_BUILDERS:
        raise KeyError(f"Unknown model key: {spec['model']}")
    if T["warmup_epochs"] >= spec["epochs"]:
        raise ValueError("warmup_epochs must be smaller than epochs")
    jobs = [(spec["model"], spec["lr"], spec["wd"], spec["epochs"], spec["batch_size"], s) for s in spec["seeds"]]
    tqdm.write(f"\n{'=' * 70}\nEXPERIMENT {name}: {run_id_for(*jobs[0][:5])}\n{'=' * 70}")
    for job in tqdm(jobs, desc=f"queue [{name}]", dynamic_ncols=True):
        run_cv(*job)
        if exp.get("regression"):
            regression_check(*job)
    summarize(name, spec)

print("Cell 10 done")


EXPERIMENT DROP_EP480: resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_dropout0.2


queue [DROP_EP480]:   0%|          | 0/3 [00:00<?, ?it/s]

SKIP resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_dropout0.2 seed 0: oof_resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_dropout0.2_s0.npz exists
SKIP resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_dropout0.2 seed 1: oof_resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_dropout0.2_s1.npz exists
SKIP resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_dropout0.2 seed 2: oof_resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_dropout0.2_s2.npz exists
SUMMARY DROP_EP480 vs resnet_32_64_128_lr0.1_wd0.002_ep240_bs32 (3 seeds, flip-averaged)
  single mean: new 0.5867 ref 0.5432 delta +4.35 points, paired SE 0.63  -> CLEAR GAIN
  ensemble:    new 0.6155 ref 0.5791 delta +3.64 points, paired SE 0.84
  rule: clear gain needs >= +2.0 single-mean points; between -2.0 and +2.0 is inconclusive

EXPERIMENT DROP_EP480_MIX: resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixmixup_dropout0.2


queue [DROP_EP480_MIX]:   0%|          | 0/3 [00:00<?, ?it/s]

SKIP resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixmixup_dropout0.2 seed 0: oof_resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixmixup_dropout0.2_s0.npz exists
SKIP resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixmixup_dropout0.2 seed 1: oof_resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixmixup_dropout0.2_s1.npz exists
SKIP resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixmixup_dropout0.2 seed 2: oof_resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixmixup_dropout0.2_s2.npz exists
SUMMARY DROP_EP480_MIX vs resnet_32_64_128_lr0.1_wd0.002_ep240_bs32_mixmixup (3 seeds, flip-averaged)
  single mean: new 0.5853 ref 0.5496 delta +3.57 points, paired SE 0.62  -> CLEAR GAIN
  ensemble:    new 0.6094 ref 0.5745 delta +3.49 points, paired SE 0.86
  rule: clear gain needs >= +2.0 single-mean points; between -2.0 and +2.0 is inconclusive

EXPERIMENT DROP_EP480_CUTOUT: resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutout_dropout0.2


queue [DROP_EP480_CUTOUT]:   0%|          | 0/3 [00:00<?, ?it/s]

SKIP resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutout_dropout0.2 seed 0: oof_resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutout_dropout0.2_s0.npz exists
SKIP resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutout_dropout0.2 seed 1: oof_resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutout_dropout0.2_s1.npz exists
SKIP resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutout_dropout0.2 seed 2: oof_resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutout_dropout0.2_s2.npz exists
SUMMARY DROP_EP480_CUTOUT vs resnet_32_64_128_lr0.1_wd0.002_ep240_bs32_mixcutout (3 seeds, flip-averaged)
  single mean: new 0.5562 ref 0.5370 delta +1.92 points, paired SE 0.63  -> inconclusive
  ensemble:    new 0.5973 ref 0.5730 delta +2.43 points, paired SE 0.82
  rule: clear gain needs >= +2.0 single-mean points; between -2.0 and +2.0 is inconclusive

EXPERIMENT DROP_EP480_CUTMIX: resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutmix_dropout0.2


queue [DROP_EP480_CUTMIX]:   0%|          | 0/3 [00:00<?, ?it/s]

SKIP resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutmix_dropout0.2 seed 0: oof_resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutmix_dropout0.2_s0.npz exists
SKIP resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutmix_dropout0.2 seed 1: oof_resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutmix_dropout0.2_s1.npz exists
SKIP resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutmix_dropout0.2 seed 2: oof_resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutmix_dropout0.2_s2.npz exists
SUMMARY DROP_EP480_CUTMIX vs resnet_32_64_128_lr0.1_wd0.002_ep240_bs32_mixcutmix (3 seeds, flip-averaged)
  single mean: new 0.5459 ref 0.5324 delta +1.35 points, paired SE 0.62  -> inconclusive
  ensemble:    new 0.5796 ref 0.5695 delta +1.01 points, paired SE 0.84
  rule: clear gain needs >= +2.0 single-mean points; between -2.0 and +2.0 is inconclusive
Cell 10 done


In [11]:
# Cell 11: Results table
metrics_path = Path(CONFIG["paths"]["metrics_csv"])
if not metrics_path.exists():
    raise FileNotFoundError(f"{metrics_path} not found; run Cell 10 first")
with open(metrics_path, newline="") as f:
    rows = list(csv.DictReader(f))

groups = {}
for r in rows:
    groups.setdefault(r["run_id"], []).append(r)

summary = []
for rid, rs in groups.items():
    seeds = sorted({int(r["seed"]) for r in rs})
    pooled, mca = [], []
    for s in seeds:
        with np.load(RES / f"oof_{rid}_s{s}.npz") as z:
            labels, preds = z["labels"], z["preds"]
        pooled.append(float((preds == labels).mean()))
        mca.append(float(np.mean([(preds[labels == c] == c).mean() for c in range(NC)])))
    fold_acc = np.array([float(r["val_acc"]) for r in rs])
    p = float(np.mean(pooled))
    summary.append({
        "run_id": rid, "params": int(rs[0]["params"]), "pooled": p,
        "se": float(np.sqrt(p * (1 - p) / len(Y_TRAIN))), "mca": float(np.mean(mca)),
        "fold_mean": float(fold_acc.mean()), "fold_std": float(fold_acc.std(ddof=1)) if len(fold_acc) > 1 else float("nan"),
        "seeds": len(seeds), "seed_std": float(np.std(pooled, ddof=1)) if len(pooled) > 1 else float("nan"),
        "sec_fold": float(np.mean([float(r["seconds"]) for r in rs])),
    })

summary.sort(key=lambda d: -d["pooled"])
print(f"{'run_id':44s} {'params':>7} {'pooled':>7} {'binomSE':>7} {'meanCls':>7} {'foldMean':>8} {'foldStd':>7} {'seeds':>5} {'seedStd':>7} {'sec/fold':>8}")
for d in summary:
    print(f"{d['run_id']:44s} {d['params']:7d} {d['pooled']:7.4f} {d['se']:7.4f} {d['mca']:7.4f} {d['fold_mean']:8.4f} {d['fold_std']:7.4f} {d['seeds']:5d} {d['seed_std']:7.4f} {d['sec_fold']:8.1f}")
print(f"Cell 11 done: {len(summary)} runs")

run_id                                        params  pooled binomSE meanCls foldMean foldStd seeds seedStd sec/fold
resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_dropout0.2  308650  0.5867  0.0111  0.5847   0.5867  0.0361     3  0.0136    166.9
resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixmixup_dropout0.2  308650  0.5853  0.0111  0.5831   0.5853  0.0214     3  0.0036    174.3
resnet_32_64_128_lr0.1_wd0.002_ep480_bs32     308650  0.5617  0.0112  0.5596   0.5618  0.0289     3  0.0145    158.1
resnet_32_64_128_lr0.1_wd0.002_ep240_bs32_dropout0.2  308650  0.5595  0.0112  0.5565   0.5596  0.0304     3  0.0091     81.8
resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutout_dropout0.2  308650  0.5562  0.0112  0.5552   0.5563  0.0352     3  0.0023    192.3
resnet_32_64_128_lr0.1_wd0.002_ep240_bs32_cj_brightness0.3_cj_contrast0.3_cj_saturation0.3  308650  0.5538  0.0112  0.5520   0.5539  0.0214     3  0.0058     90.0
resnet_32_64_128_lr0.1_wd0.002_ep240_bs32_mixmixup_mixup_alpha0.2  308650  0.5494  0

In [12]:
# Cell 13: OOF registry, run inventory and focus-run analysis (read-only)
import math
import re

OOF_DIR = Path(CONFIG["paths"]["results_dir"])
_D = CONFIG.get("defaults", {"model": "resnet_32_64_128", "lr": 0.1, "wd": 0.002, "epochs": 240, "batch_size": 32})
OOF_BASE = f"{_D['model']}_lr{_D['lr']:g}_wd{_D['wd']:g}_ep{_D['epochs']}_bs{_D['batch_size']}"
OOF_CANON = {
    "reference": OOF_BASE,
    "mixup": OOF_BASE + "_mixmixup",
    "cutout": OOF_BASE + "_mixcutout",
    "cutmix": OOF_BASE + "_mixcutmix",
}
OOF_N = len(Y_TRAIN)
FOCUS = None


def refresh_oof(res_dir=None):
    d = OOF_DIR if res_dir is None else Path(res_dir)
    runs = {}
    for p in sorted(d.glob("oof_*.npz")):
        m = re.match(r"^oof_(.+)_s(\d+)\.npz$", p.name)
        if m is None:
            continue
        rid, seed = m.group(1), int(m.group(2))
        try:
            with np.load(p) as z:
                lab, fid, lg, cfg = z["labels"], z["fold_id"], z["logits"].astype(np.float64), json.loads(str(z["config"]))
        except Exception as exc:
            print(f"WARN cannot read {p.name}: {exc}")
            continue
        if not (np.array_equal(lab, Y_TRAIN) and np.array_equal(fid, FOLD_ID) and lg.shape == (OOF_N, NC) and np.isfinite(lg).all()):
            print(f"WARN skipped {p.name}: labels, folds or logits do not match the current data")
            continue
        r = runs.setdefault(rid, {
            "id": rid, "cfg": cfg, "model": cfg.get("model"), "mix": cfg.get("augmentation", {}).get("mix"),
            "flip": cfg.get("flip_avg", "not recorded"), "logits": {}, "mtime": 0.0,
        })
        r["logits"][seed] = lg
        r["mtime"] = max(r["mtime"], p.stat().st_mtime)
    return runs


def oof_hits(lg):
    return (lg.argmax(axis=1) == Y_TRAIN).astype(np.float64)


def oof_ens_hits(arrs):
    return oof_hits(np.mean(arrs, axis=0))


def oof_paired(a, b):
    d = b - a
    return d.mean() * 100, d.std(ddof=1) / np.sqrt(len(d)) * 100


def oof_softmax(a):
    e = np.exp(a - a.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def oof_params(model):
    return CONFIG["expected_params"].get(model, 0)


def oof_verdict(delta):
    return "CLEAR GAIN" if delta >= 2.0 else ("clearly worse" if delta <= -2.0 else "inconclusive")


def comb_ok(n, k, limit=5000):
    return math.comb(n, k) <= limit


def run_seeds(r):
    return sorted(r["logits"])


def run_hits_mean(r, seeds):
    return np.mean([oof_hits(r["logits"][s]) for s in seeds], axis=0)


def run_ens_hits(r, seeds):
    return oof_ens_hits([r["logits"][s] for s in seeds])


def ref_id_for(r):
    return OOF_BASE if r["mix"] == "none" else f"{OOF_BASE}_mix{r['mix']}"


def canon_dict(runs):
    out = {}
    for name, rid in OOF_CANON.items():
        r = runs.get(rid)
        if r is not None:
            for s in run_seeds(r):
                out[(name, s)] = r["logits"][s]
    return out


def pick_focus(runs, focus=None):
    pool = list(runs.values())
    if focus is not None:
        pool = [r for r in pool if focus in r["id"]]
    else:
        canon = set(OOF_CANON.values())
        non = [r for r in pool if r["id"] not in canon]
        pool = non if non else pool
    return max(pool, key=lambda r: r["mtime"]) if pool else None


def ens_row(label, arrs, params):
    single = np.mean([oof_hits(a) for a in arrs], axis=0)
    cols = []
    for mode in ("logit", "prob"):
        avg = np.mean(arrs, axis=0) if mode == "logit" else np.mean([oof_softmax(a) for a in arrs], axis=0)
        d = oof_hits(avg) - single
        se = d.std(ddof=1) / np.sqrt(len(d))
        cols.append(f"{single.mean() + d.mean():.4f} {d.mean() * 100:+6.2f} {se * 100:5.2f}")
    print(f"{label:34s} {len(arrs):3d} {params:10,d} {single.mean():.4f} | {cols[0]} | {cols[1]}")


OOF_RUNS = refresh_oof()
print(f"found {sum(len(r['logits']) for r in OOF_RUNS.values())} OOF files in {len(OOF_RUNS)} runs under {OOF_DIR}")
print(f"\nrun (suffix after {OOF_BASE})".ljust(58) + f"{'model':<18s}{'mix':<8s}{'flip':<14s}{'seeds':<8s}{'single':>7s}{'ens':>8s}")
for r in sorted(OOF_RUNS.values(), key=lambda x: x["mtime"]):
    sd = run_seeds(r)
    short = r["id"][len(OOF_BASE):] if r["id"].startswith(OOF_BASE) else r["id"]
    short = short or "(reference)"
    print(f"{short:<57s}{str(r['model']):<18s}{str(r['mix']):<8s}{str(r['flip']):<14s}{','.join(map(str, sd)):<8s}{run_hits_mean(r, sd).mean():7.4f}{run_ens_hits(r, sd).mean():8.4f}")

f13 = pick_focus(OOF_RUNS, FOCUS)
if f13 is None:
    print("\nno OOF runs found; nothing to analyse")
else:
    sd = run_seeds(f13)
    print(f"\nFOCUS run: {f13['id']}  (seeds {sd}, flip_avg recorded: {f13['flip']})")
    if len(sd) < 3:
        print(f"  partial run: {len(sd)} seed(s) finished; conclusions need all seeds")
    print(f"{'group':34s} {'n':>3} {'params':>10} {'single':>6} | {'logit-avg: acc':>14} {'gain':>6} {'SE':>5} | {'prob-avg: acc':>13} {'gain':>6} {'SE':>5}")
    ens_row(f"focus ({len(sd)} seeds)", [f13["logits"][s] for s in sd], len(sd) * oof_params(f13["model"]))
    ref13 = OOF_RUNS.get(ref_id_for(f13))
    if ref13 is None:
        print(f"  reference run {ref_id_for(f13)} not found; no comparison")
    elif ref13 is f13:
        print("  focus is the reference run; no comparison")
    else:
        common = [s for s in sd if s in ref13["logits"]]
        if not common:
            print("  no common seeds with the reference; no comparison")
        else:
            ens_row(f"reference ({len(common)} seeds)", [ref13["logits"][s] for s in common], len(common) * oof_params(ref13["model"]))
            dl, se = oof_paired(run_hits_mean(ref13, common), run_hits_mean(f13, common))
            de, se_e = oof_paired(run_ens_hits(ref13, common), run_ens_hits(f13, common))
            print(f"\nFOCUS vs REFERENCE on seeds {common}")
            print(f"  single mean: delta {dl:+.2f} points, paired SE {se:.2f}  -> {oof_verdict(dl)}")
            print(f"  ensemble:    delta {de:+.2f} points, paired SE {se_e:.2f}")
            print("  rule: clear gain needs >= +2.0 single-mean points; between -2.0 and +2.0 is inconclusive")
            if str(ref13["flip"]) != str(f13["flip"]):
                print(f"  NOTE flip_avg recorded: reference {ref13['flip']}, focus {f13['flip']}; check both sides use the same flip setting")
    arrs = [f13["logits"][s] for s in sd]
    print("\nensemble size curve over the focus run's seeds (mean and std of pooled OOF acc over subsets)")
    for k in range(1, len(arrs) + 1):
        if not comb_ok(len(arrs), k):
            continue
        accs = [oof_ens_hits(list(c)).mean() for c in itertools.combinations(arrs, k)]
        print(f"k={k:2d}  subsets={len(accs):4d}  mean {np.mean(accs):.4f}  std {np.std(accs):.4f}")
print("Cell 13 done")

found 70 OOF files in 24 runs under results

run (suffix after resnet_32_64_128_lr0.1_wd0.002_ep240_bs32)model             mix     flip          seeds    single     ens
_area_min0.6_aspect_max1.25_rot_deg15.0                  resnet_32_64_128  none    True          0,1,2    0.5053  0.5220
_reg                                                     resnet_32_64_128  none    True          0        0.5493  0.5493
resnet_32_64_128_lr0.1_wd0.002_ep480_bs32                resnet_32_64_128  none    True          0,1,2    0.5617  0.5988
_mixmixup                                                resnet_32_64_128  mixup   not recorded  0,1,2    0.5496  0.5745
(reference)                                              resnet_32_64_128  none    not recorded  0,1,2    0.5432  0.5791
_mixcutout                                               resnet_32_64_128  cutout  not recorded  0,1,2    0.5370  0.5730
_mixcutmix                                               resnet_32_64_128  cutmix  not recorded  0,1,2   

In [13]:
# Cell 14: Per-class error analysis of the focus run (read-only)
if "refresh_oof" not in globals():
    raise RuntimeError("Run Cell 13 once in this kernel first (it defines the OOF helpers)")
OOF_RUNS = refresh_oof()


def lab14(c):
    return LABELS_RAW[c] if c < len(LABELS_RAW) else str(c)


def analyse14(run):
    seeds = run_seeds(run)
    ens_logits = np.mean([run["logits"][s] for s in seeds], axis=0)
    ens_preds = ens_logits.argmax(axis=1)
    ens_ok = ens_preds == Y_TRAIN
    singles = [(s, float(oof_hits(run["logits"][s]).mean())) for s in seeds]
    best_s, best_acc = max(singles, key=lambda t: t[1])
    n = len(Y_TRAIN)
    print(f"focus run: {run['id']}  (flip_avg recorded: {run['flip']})")
    print(f"{len(seeds)}-member ensemble OOF acc: {ens_ok.mean():.4f}")
    print(f"best single seed: {best_s}  OOF acc: {best_acc:.4f}\n")

    print("=" * 90)
    print(f"PER-CLASS METRICS ({len(seeds)}-member ensemble, logit averaging)")
    print("=" * 90)
    print(f"{'Class':>3} {'Breed':<28s} {'Support':>7} {'Accuracy':>8} {'Precision':>9} {'Recall':>6} {'F1':>6}")
    print("-" * 90)
    rows = []
    for c in range(NC):
        mt, mp = Y_TRAIN == c, ens_preds == c
        support = int(mt.sum())
        tp, fp = int((mt & mp).sum()), int((~mt & mp).sum())
        recall = tp / support if support else 0.0
        precision = tp / (tp + fp) if tp + fp else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append((c, lab14(c), support, recall, precision, f1))
        print(f"{c:>3} {lab14(c):<28s} {support:>7} {recall:>8.4f} {precision:>9.4f} {recall:>6.4f} {f1:>6.4f}")

    print(f"\nBreeds sorted by WORST accuracy ({len(seeds)}-member ensemble):")
    for c, name, sup, rec, prec, f1 in sorted(rows, key=lambda x: x[3]):
        print(f"  {c:>2} {name:<28s} acc={rec:.4f}  prec={prec:.4f}  f1={f1:.4f}  (n={sup})")

    print("\n" + "=" * 90)
    print(f"CONFUSION MATRIX (rows=true, cols=predicted)")
    print("=" * 90)
    conf = np.zeros((NC, NC), dtype=int)
    for i in range(n):
        conf[Y_TRAIN[i], ens_preds[i]] += 1
    print("     " + " ".join(f"{c:>5}" for c in range(NC)))
    for c in range(NC):
        print(f"{c:>3}  " + " ".join(f"{conf[c, j]:>5}" for j in range(NC)) + f"  {lab14(c)}")

    print("\n" + "=" * 90)
    print("TOP 20 CONFUSION PAIRS (true -> predicted)")
    print("=" * 90)
    pairs = [(conf[a, b], a, b) for a in range(NC) for b in range(NC) if a != b and conf[a, b] > 0]
    pairs.sort(key=lambda x: -x[0])
    for count, a, b in pairs[:20]:
        print(f"  {count:>4} errors ({count / conf[a].sum() * 100:>5.1f}% of {lab14(a):<28s}) misclassified as {lab14(b)}")

    print("\n" + "=" * 90)
    print("TOP 15 SYMMETRIC CONFUSION PAIRS (both directions summed)")
    print("=" * 90)
    sym = {(a, b): conf[a, b] + conf[b, a] for a in range(NC) for b in range(a + 1, NC)}
    for (a, b), total in sorted(sym.items(), key=lambda x: -x[1])[:15]:
        print(f"  {total:>4} errors: {lab14(a):<28s} <-> {lab14(b)}")

    probs = oof_softmax(ens_logits)
    print(f"\n{'Class':>3} {'Breed':<28s} {'Correct conf':>12} {'Wrong conf':>10} {'Margin':>8}")
    print("-" * 75)
    for c in range(NC):
        mask = Y_TRAIN == c
        cm, wm = mask & ens_ok, mask & ~ens_ok
        cc = probs[cm, c].mean() if cm.any() else float("nan")
        wc = probs[wm, c].mean() if wm.any() else float("nan")
        mg = (probs[wm, c] - probs[wm].max(axis=1)).mean() if wm.any() else float("nan")
        print(f"{c:>3} {lab14(c):<28s} {cc:>12.4f} {wc:>10.4f} {mg:>8.4f}")

    ref = OOF_RUNS.get(ref_id_for(run))
    if ref is not None and ref is not run:
        common = [s for s in seeds if s in ref["logits"]]
        if common:
            ok_new = run_ens_hits(run, common).astype(bool)
            ok_ref = run_ens_hits(ref, common).astype(bool)
            print("\n" + "=" * 90)
            print(f"PER-CLASS ENSEMBLE ACCURACY vs REFERENCE ({ref['id']}, seeds {common})")
            print("=" * 90)
            print(f"{'Class':>3} {'Breed':<28s} {'ref':>8} {'focus':>8} {'delta':>8}")
            for c in range(NC):
                mask = Y_TRAIN == c
                a, b = ok_ref[mask].mean(), ok_new[mask].mean()
                print(f"{c:>3} {lab14(c):<28s} {a:>8.4f} {b:>8.4f} {(b - a) * 100:>+8.2f}")
            print(f"overall ref {ok_ref.mean():.4f} focus {ok_new.mean():.4f}")


f14 = pick_focus(OOF_RUNS, FOCUS)
if f14 is None:
    print("no OOF runs found; nothing to analyse")
else:
    analyse14(f14)
print("Cell 14 done")

focus run: resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutmix_dropout0.2  (flip_avg recorded: True)
3-member ensemble OOF acc: 0.5796
best single seed: 2  OOF acc: 0.5488

PER-CLASS METRICS (3-member ensemble, logit averaging)
Class Breed                        Support Accuracy Precision Recall     F1
------------------------------------------------------------------------------------------
  0 0	Bedlington_terrier             217   0.6959    0.7123 0.6959 0.7040
  1 1	Boston_bull                    216   0.7731    0.6627 0.7731 0.7137
  2 2	English_setter                 195   0.5282    0.4578 0.5282 0.4905
  3 3	Italian_greyhound              220   0.3909    0.4300 0.3909 0.4095
  4 4	Labrador_retriever             204   0.4608    0.4796 0.4608 0.4700
  5 5	Norfolk_terrier                204   0.6716    0.5880 0.6716 0.6270
  6 6	Walker_hound                   178   0.6742    0.7317 0.6742 0.7018
  7 7	boxer                          180   0.5222    0.6309 0.5222 0.5714
  8 8	golden_

In [14]:
# Cell 15: Canonical OOF ensemble (read-only)
if "refresh_oof" not in globals():
    raise RuntimeError("Run Cell 13 once in this kernel first (it defines the OOF helpers)")
OOF_RUNS = refresh_oof()


def correct15(L, members):
    return oof_ens_hits([L[m] for m in members])


def report15(L, label, members, pm):
    single = np.mean([correct15(L, [m]) for m in members], axis=0)
    d = correct15(L, members) - single
    se = d.std(ddof=1) / np.sqrt(len(d))
    print(f"{label:24s} {len(members):3d} {len(members) * pm:10,d} single {single.mean():.4f} | logit-avg {single.mean() + d.mean():.4f} gain {d.mean() * 100:+.2f} SE {se * 100:.2f}")


def analyse15():
    LF = canon_dict(OOF_RUNS)
    if not LF:
        print(f"no canonical runs found (expected run ids starting with {OOF_BASE}); nothing to analyse")
        return
    pm = oof_params(_D["model"])
    all_m = list(LF)
    groups = sorted({m[0] for m in all_m}, key=list(OOF_CANON).index)
    flips = sorted({str(OOF_RUNS[OOF_CANON[g]]["flip"]) for g in groups})
    print(f"CANONICAL OOF from {OOF_DIR}: {len(all_m)} members, groups {groups}, flip_avg recorded: {flips}")
    for g in groups:
        ms = [m for m in all_m if m[0] == g]
        report15(LF, f"{g} ({len(ms)} seeds)", ms, pm)
    pair = [m for m in all_m if m[0] in ("reference", "mixup")]
    if len({m[0] for m in pair}) == 2:
        report15(LF, "reference+mixup", pair, pm)
    report15(LF, f"all {len(groups)} recipes", all_m, pm)

    print("\nensemble size curve, logit averaging (mean and std of pooled OOF acc over all subsets)")
    for k in sorted({1, 2, 3, 4, 6, len(all_m)}):
        if k > len(all_m) or not comb_ok(len(all_m), k):
            continue
        accs = [correct15(LF, list(c)).mean() for c in itertools.combinations(all_m, k)]
        print(f"k={k:2d}  subsets={len(accs):4d}  mean {np.mean(accs):.4f}  std {np.std(accs):.4f}  params {k * pm:,}")

    nf_dir = CONFIG["paths"].get("noflip_results_dir")
    if nf_dir is None or not Path(nf_dir).exists():
        print("\nno-flip comparison skipped: CONFIG['paths']['noflip_results_dir'] not set or missing")
        return
    LN = canon_dict(refresh_oof(nf_dir))
    common = [m for m in all_m if m in LN]
    if not common:
        print(f"\nno-flip comparison skipped: no common canonical members in {nf_dir}")
        return
    same = sum(np.array_equal(LF[m], LN[m]) for m in common)
    print(f"\nNO-FLIP OOF from {nf_dir}; members with identical logits in both folders: {same}/{len(common)}")
    for g in groups:
        ms = [m for m in common if m[0] == g]
        if ms:
            a = np.mean([correct15(LN, [m]).mean() for m in ms])
            b = np.mean([correct15(LF, [m]).mean() for m in ms])
            print(f"{g:12s} single no-flip {a:.4f} flip {b:.4f} diff {(b - a) * 100:+.2f}")
    a, b = correct15(LN, common), correct15(LF, common)
    d = b - a
    print(f"all common members: no-flip {a.mean():.4f} flip {b.mean():.4f} diff {d.mean() * 100:+.2f} paired SE {d.std(ddof=1) / np.sqrt(len(d)) * 100:.2f}")


analyse15()
print("Cell 15 done")

CANONICAL OOF from results: 12 members, groups ['reference', 'mixup', 'cutout', 'cutmix'], flip_avg recorded: ['not recorded']
reference (3 seeds)        3    925,950 single 0.5432 | logit-avg 0.5791 gain +3.59 SE 0.52
mixup (3 seeds)            3    925,950 single 0.5496 | logit-avg 0.5745 gain +2.49 SE 0.51
cutout (3 seeds)           3    925,950 single 0.5370 | logit-avg 0.5730 gain +3.60 SE 0.51
cutmix (3 seeds)           3    925,950 single 0.5324 | logit-avg 0.5695 gain +3.71 SE 0.49
reference+mixup            6  1,851,900 single 0.5464 | logit-avg 0.5907 gain +4.43 SE 0.55
all 4 recipes             12  3,703,800 single 0.5406 | logit-avg 0.5993 gain +5.87 SE 0.55

ensemble size curve, logit averaging (mean and std of pooled OOF acc over all subsets)
k= 1  subsets=  12  mean 0.5406  std 0.0085  params 308,650
k= 2  subsets=  66  mean 0.5696  std 0.0089  params 617,300
k= 3  subsets= 220  mean 0.5824  std 0.0074  params 925,950
k= 4  subsets= 495  mean 0.5885  std 0.0067  params 1

In [15]:
# Cell 16: Architecture-diversity analysis on existing OOF files (read-only)
if "refresh_oof" not in globals():
    raise RuntimeError("Run Cell 13 once in this kernel first (it defines the OOF helpers)")
OOF_RUNS = refresh_oof()
LOGGED16 = {"ref_noflip_ens3_mixup": 0.5695, "ref_noflip_ens12_four_recipes": 0.5993, "ref_flip_ens6_ref_plus_mixup": 0.5907}


def analyse16():
    suffix = f"_lr{_D['lr']:g}_wd{_D['wd']:g}_ep{_D['epochs']}_bs{_D['batch_size']}_mixmixup"
    archs = sorted(r["model"] for r in OOF_RUNS.values() if r["model"] != _D["model"] and r["id"] == f"{r['model']}{suffix}")
    if not archs:
        print(f"no architecture runs found (expected run ids like <model>{suffix}); nothing to analyse")
        return
    ALL = {}
    flips = {"arch": set(), "reference": set(), "mixup": set()}
    for a in archs:
        r = OOF_RUNS[f"{a}{suffix}"]
        flips["arch"].add(str(r["flip"]))
        for s in run_seeds(r):
            ALL[(a, "mixup", s)] = r["logits"][s]
    arch_members = list(ALL)
    base_model = _D["model"]
    ref3, mix3 = [], []
    for key, mix, store in (("reference", "none", ref3), ("mixup", "mixup", mix3)):
        r = OOF_RUNS.get(OOF_CANON[key])
        if r is not None:
            flips[key].add(str(r["flip"]))
            for s in run_seeds(r):
                ALL[(base_model, mix, s)] = r["logits"][s]
                store.append((base_model, mix, s))
    print(f"loaded {len(arch_members)} architecture members from {archs}; reference members {len(ref3)}, mixup members {len(mix3)}")
    print("flip_avg recorded in OOF configs:", {k: sorted(v) for k, v in flips.items() if v})
    print("('not recorded' means the flip status is unverified; compare like with like only)")

    def acc_vec(members):
        return oof_ens_hits([ALL[m] for m in members])

    def n_params(members):
        return sum(oof_params(m[0]) for m in members)

    def report16(label, members, logged=None, logged_name=""):
        singles = np.mean([acc_vec([m]) for m in members], axis=0)
        ens = acc_vec(members)
        d = ens - singles
        se = d.std(ddof=1) / np.sqrt(len(d))
        print(f"{label:40s} {len(members):3d} {n_params(members):10,d} single {singles.mean():.4f} | logit-avg {ens.mean():.4f} gain {d.mean() * 100:+.2f} SE {se * 100:.2f}")
        if logged is not None:
            p = ens.mean()
            se_u = np.sqrt(p * (1 - p) / len(Y_TRAIN) + logged * (1 - logged) / len(Y_TRAIN))
            print(f"    vs logged {logged_name} {logged:.4f}: {(p - logged) * 100:+.2f} points (unpaired binomial SE {se_u * 100:.2f}, conservative)")

    def paired16(label, ma, mb):
        d = acc_vec(ma) - acc_vec(mb)
        print(f"    {label}: {d.mean() * 100:+.2f} points (paired SE {d.std(ddof=1) / np.sqrt(len(d)) * 100:.2f})")

    print("\n(a) per-architecture ensembles, mixup, one run per architecture")
    print(f"{'':40s} {'n':>3s} {'params':>10s}")
    for a in archs:
        mem = [m for m in arch_members if m[0] == a]
        report16(f"{a} ({len(mem)} seeds)", mem, LOGGED16["ref_noflip_ens3_mixup"], "no-flip reference 3-seed ensemble (mixup)")

    print("\n(b) all architectures together")
    report16(f"all architectures ({len(arch_members)})", arch_members, LOGGED16["ref_noflip_ens12_four_recipes"], "no-flip 12-member reference ensemble (four recipes, not like-for-like)")
    print("pairs of architectures:")
    for a, b in itertools.combinations(archs, 2):
        report16(f"{a} + {b}", [m for m in arch_members if m[0] in (a, b)])
    print("size curve over the architecture members (mean and std of pooled OOF acc over all subsets):")
    n = len(arch_members)
    for k in sorted({1, 2, 3, 4, 6, n}):
        if k > n or not comb_ok(n, k):
            continue
        accs, prms = [], []
        for c in itertools.combinations(arch_members, k):
            accs.append(acc_vec(list(c)).mean())
            prms.append(n_params(c))
        print(f"k={k:2d}  subsets={len(accs):4d}  mean {np.mean(accs):.4f}  std {np.std(accs):.4f}  mean params {np.mean(prms):,.0f}")

    if ref3 and mix3:
        base6 = ref3 + mix3
        print("\n(c) CONFOUNDED mixed ensembles: check the recorded flip_avg values above before reading these")
        report16("reference(none) + mixup members", base6, LOGGED16["ref_flip_ens6_ref_plus_mixup"], "flip-avg ref+mixup (RESULT-15)")
        for a in archs:
            mem = ref3 + [m for m in arch_members if m[0] == a]
            report16(f"reference(none) + {a}", mem)
            paired16("minus reference+mixup baseline", mem, base6)
        mem = ref3 + arch_members
        report16("reference(none) + all architectures", mem)
        paired16("minus reference+mixup baseline", mem, base6)
    else:
        print("\n(c) skipped: canonical reference and mixup runs are not both present")


analyse16()
print("Cell 16 done")

no architecture runs found (expected run ids like <model>_lr0.1_wd0.002_ep240_bs32_mixmixup); nothing to analyse
Cell 16 done


In [16]:
# Cell 17: Member weighting and drop-one analysis of the canonical ensemble (read-only)
if "refresh_oof" not in globals():
    raise RuntimeError("Run Cell 13 once in this kernel first (it defines the OOF helpers)")
OOF_RUNS = refresh_oof()


def analyse17():
    LF = canon_dict(OOF_RUNS)
    names = list(LF)
    M = len(names)
    if M < 3:
        print(f"only {M} canonical member(s) found; need at least 3")
        return
    L = np.stack([LF[m] for m in names])
    LAB = [f"{g}_s{s}" for g, s in names]
    groups = sorted({g for g, s in names}, key=list(OOF_CANON).index)
    seeds = sorted({s for g, s in names})
    print("Rules fixed in advance:")
    print("  member/group/weighting change is notable only if")
    print("  |paired delta| >= 1.0 point AND >= 2 paired SE")
    print(f"loaded {M} canonical members, params each {oof_params(_D['model']):,}")

    def hit_w(w):
        w = np.asarray(w, dtype=np.float64)
        avg = np.tensordot(w / w.sum(), L, axes=(0, 0))
        return (avg.argmax(axis=1) == Y_TRAIN).astype(np.float64)

    def flag(delta, se):
        return "NOTABLE" if abs(delta) >= 1.0 and abs(delta) >= 2 * se else ""

    base = hit_w(np.ones(M))
    print(f"equal-weight {M}-member OOF acc {base.mean():.4f}")
    if M == 12 and abs(base.mean() - 0.5993) > 0.0005:
        print("WARN: differs from the logged 0.5993 for the 12-member set; the canonical members may have changed")

    print("\n(1) drop-one member (delta = remaining members minus all members, points)")
    print(f"{'member':<14}{'single':>8}{'drop1':>8}{'delta':>8}{'SE':>6}  flag")
    for i in range(M):
        w = np.ones(M)
        w[i] = 0
        h = hit_w(w)
        single = float((L[i].argmax(axis=1) == Y_TRAIN).mean())
        dl, se = oof_paired(base, h)
        print(f"{LAB[i]:<14}{single:8.4f}{h.mean():8.4f}{dl:+8.2f}{se:6.2f}  {flag(dl, se)}")

    print("\n(2) drop a whole recipe")
    for g in groups:
        w = np.array([0.0 if n == g else 1.0 for n, s in names])
        if w.sum() == 0:
            continue
        h = hit_w(w)
        dl, se = oof_paired(base, h)
        print(f"drop {g:<10}{h.mean():8.4f}{dl:+8.2f}{se:6.2f}  {flag(dl, se)}")

    print("\n(3) drop a whole seed")
    for seed in seeds:
        w = np.array([0.0 if s == seed else 1.0 for n, s in names])
        if w.sum() == 0:
            continue
        h = hit_w(w)
        dl, se = oof_paired(base, h)
        print(f"drop seed {seed:<6}{h.mean():8.4f}{dl:+8.2f}{se:6.2f}  {flag(dl, se)}")

    def nll(avg, y):
        z = avg - avg.max(axis=1, keepdims=True)
        lse = np.log(np.exp(z).sum(axis=1))
        return float((lse - z[np.arange(len(y)), y]).mean())

    steps = int(round(1.25 * M))

    def caruana(tr):
        counts = np.zeros(M)
        for _ in range(steps):
            best, best_j = None, None
            for j in range(M):
                c = counts.copy()
                c[j] += 1
                avg = np.tensordot(c / c.sum(), L[:, tr], axes=(0, 0))
                v = nll(avg, Y_TRAIN[tr])
                if best is None or v < best:
                    best, best_j = v, j
            counts[best_j] += 1
        return counts

    print(f"\n(4) fold-nested greedy weighting ({steps} picks, with replacement, NLL)")
    nested = np.zeros(len(Y_TRAIN))
    all_counts = []
    for f in sorted(set(FOLD_ID.tolist())):
        va = np.where(FOLD_ID == f)[0]
        tr = np.where(FOLD_ID != f)[0]
        c = caruana(tr)
        all_counts.append(c)
        avg = np.tensordot(c / c.sum(), L[:, va], axes=(0, 0))
        nested[va] = (avg.argmax(axis=1) == Y_TRAIN[va]).astype(np.float64)
    dl, se = oof_paired(base, nested)
    print(f"nested greedy acc {nested.mean():.4f} vs equal {base.mean():.4f}")
    print(f"delta {dl:+.2f} points, paired SE {se:.2f}  {flag(dl, se)}")
    cnt = np.array(all_counts)
    print(f"\nmean picks per member across folds (equal weight = {steps / M:.2f}), min..max")
    for i in range(M):
        print(f"{LAB[i]:<14}{cnt[:, i].mean():6.2f}  {cnt[:, i].min():.0f}..{cnt[:, i].max():.0f}")


analyse17()
print("Cell 17 done")

Rules fixed in advance:
  member/group/weighting change is notable only if
  |paired delta| >= 1.0 point AND >= 2 paired SE
loaded 12 canonical members, params each 308,650
equal-weight 12-member OOF acc 0.5993

(1) drop-one member (delta = remaining members minus all members, points)
member          single   drop1   delta    SE  flag
reference_s0    0.5493  0.6003   +0.10  0.34  
reference_s1    0.5387  0.5993   +0.00  0.31  
reference_s2    0.5417  0.5978   -0.15  0.29  
mixup_s0        0.5422  0.6023   +0.30  0.23  
mixup_s1        0.5568  0.6023   +0.30  0.21  
mixup_s2        0.5498  0.6018   +0.25  0.21  
cutout_s0       0.5457  0.6059   +0.66  0.35  
cutout_s1       0.5366  0.6033   +0.40  0.35  
cutout_s2       0.5285  0.5978   -0.15  0.35  
cutmix_s0       0.5270  0.5993   +0.00  0.25  
cutmix_s1       0.5356  0.5983   -0.10  0.23  
cutmix_s2       0.5346  0.5993   +0.00  0.23  

(2) drop a whole recipe
drop reference   0.5988   -0.05  0.47  
drop mixup       0.6038   +0.45  0

In [17]:
# Cell 18: Do candidate members add to the canonical ensemble? (read-only)
if "refresh_oof" not in globals():
    raise RuntimeError("Run Cell 13 once in this kernel first (it defines the OOF helpers)")
OOF_RUNS = refresh_oof()
LOGGED18 = {"canonical_12": 0.5993}


def analyse18():
    canon = canon_dict(OOF_RUNS)
    if len(canon) < 2:
        print(f"only {len(canon)} canonical member(s) found; nothing to compare against")
        return
    cand = pick_focus(OOF_RUNS, FOCUS)
    if cand is None or cand["id"] in OOF_CANON.values():
        print("no non-canonical run found to test")
        return
    pm = oof_params(_D["model"])
    base_arrs = list(canon.values())
    base_h = oof_ens_hits(base_arrs)
    sd = run_seeds(cand)
    cand_arrs = [cand["logits"][s] for s in sd]
    cand_pm = oof_params(cand["model"])

    print("Rule fixed in advance:")
    print("  include candidate members in the final ensemble only if the paired delta")
    print("  (canonical members + candidate members, minus canonical members) is >= +0.50 points AND >= 1.5 paired SE")
    print("  per-run rows are descriptive only (not for selection)")
    print(f"\ncandidate run: {cand['id']} (seeds {sd}, model {cand['model']}, flip_avg recorded: {cand['flip']})")
    if len(sd) < 3:
        print(f"  partial run: {len(sd)} seed(s) finished; conclusions need all seeds")
    print(f"canonical flip_avg recorded: {sorted({str(OOF_RUNS[rid]['flip']) for rid in OOF_CANON.values() if rid in OOF_RUNS})}")

    print(f"\nCANONICAL: {len(base_arrs)} members: acc {base_h.mean():.4f}, params {len(base_arrs) * pm:,}")
    if len(base_arrs) == 12 and abs(base_h.mean() - LOGGED18["canonical_12"]) > 0.0005:
        print("WARN: differs from the logged 0.5993 for the 12-member set; the canonical members may have changed")

    h = oof_ens_hits(base_arrs + cand_arrs)
    dl, se = oof_paired(base_h, h)
    met = dl >= 0.5 and dl >= 1.5 * se
    print(f"\nPRIMARY: canonical + candidate ({len(base_arrs) + len(cand_arrs)} members): acc {h.mean():.4f}, params {len(base_arrs) * pm + len(cand_arrs) * cand_pm:,}")
    print(f"  paired delta {dl:+.2f} points, paired SE {se:.2f}, ratio {dl / se if se else float('nan'):+.2f}  ->  {'RULE MET' if met else 'rule not met'}")

    h_alone = oof_ens_hits(cand_arrs)
    d2, se2 = oof_paired(base_h, h_alone)
    print(f"\nREFERENCE: candidate alone ({len(cand_arrs)} members): acc {h_alone.mean():.4f}, delta vs canonical {d2:+.2f}, paired SE {se2:.2f}")

    print("\nDESCRIPTIVE: canonical + each other non-canonical run")
    others = [r for r in OOF_RUNS.values() if r["id"] not in OOF_CANON.values()]
    for r in sorted(others, key=lambda x: x["mtime"]):
        rs = run_seeds(r)
        hh = oof_ens_hits(base_arrs + [r["logits"][s] for s in rs])
        d, s_ = oof_paired(base_h, hh)
        short = r["id"][len(OOF_BASE):] if r["id"].startswith(OOF_BASE) else r["id"]
        print(f"  + {short or '(reference)':<60s} seeds {len(rs)} acc {hh.mean():.4f}  delta {d:+.2f}  paired SE {s_:.2f}")

    print("\nCONTROL: size effect without new members")
    rng = np.random.default_rng(0)
    k = max(2, len(base_arrs) * 2 // 3)
    accs = [oof_ens_hits([base_arrs[i] for i in rng.choice(len(base_arrs), size=k, replace=False)]).mean() for _ in range(200)]
    print(f"  random {k}-of-{len(base_arrs)} canonical subsets: mean acc {np.mean(accs):.4f}, std {np.std(accs):.4f}")
    print("  (shows how much accuracy moves from membership alone)")


analyse18()
print("Cell 18 done")

Rule fixed in advance:
  include candidate members in the final ensemble only if the paired delta
  (canonical members + candidate members, minus canonical members) is >= +0.50 points AND >= 1.5 paired SE
  per-run rows are descriptive only (not for selection)

candidate run: resnet_32_64_128_lr0.1_wd0.002_ep480_bs32_mixcutmix_dropout0.2 (seeds [0, 1, 2], model resnet_32_64_128, flip_avg recorded: True)
canonical flip_avg recorded: ['not recorded']

CANONICAL: 12 members: acc 0.5993, params 3,703,800

PRIMARY: canonical + candidate (15 members): acc 0.6023, params 4,629,750
  paired delta +0.30 points, paired SE 0.28, ratio +1.10  ->  rule not met

REFERENCE: candidate alone (3 members): acc 0.5796, delta vs canonical -1.97, paired SE 0.82

DESCRIPTIVE: canonical + each other non-canonical run
  + _area_min0.6_aspect_max1.25_rot_deg15.0                      seeds 3 acc 0.6059  delta +0.66  paired SE 0.48
  + _reg                                                         seeds 1 acc 0.605

In [18]:
# Cell 19: 12-member ensemble at 480 epochs + dropout 0.2 vs the canonical 12 (read-only)
RES19 = Path(CONFIG["paths"]["results_dir"])
BASE19 = "resnet_32_64_128_lr0.1_wd0.002"
SEEDS19 = (0, 1, 2)
RECIPES19 = [("none", ""), ("mixup", "_mixmixup"), ("cutout", "_mixcutout"), ("cutmix", "_mixcutmix")]
LOGGED19 = 0.5993


def load19(rid):
    out = {}
    for s in SEEDS19:
        p = RES19 / f"oof_{rid}_s{s}.npz"
        if not p.exists():
            continue
        with np.load(p) as z:
            lg = z["logits"].astype(np.float64)
            labels, fold_id = z["labels"], z["fold_id"]
        if not np.array_equal(labels, Y_TRAIN) or not np.array_equal(fold_id, FOLD_ID):
            raise ValueError(f"{p.name}: labels or fold_id differ from Y_TRAIN / FOLD_ID")
        if lg.shape != (len(Y_TRAIN), NC) or not np.isfinite(lg).all():
            raise ValueError(f"{p.name}: unexpected logits shape or non-finite values")
        out[s] = lg
    return out


def hits19(arrs):
    return (np.mean(arrs, axis=0).argmax(axis=1) == Y_TRAIN).astype(np.float64)


def paired19(base, other):
    d = other - base
    return d.mean() * 100, d.std(ddof=1) / np.sqrt(len(d)) * 100


REF19 = {n: load19(f"{BASE19}_ep240_bs32{t}") for n, t in RECIPES19}
NEW19 = {n: load19(f"{BASE19}_ep480_bs32{t}_dropout0.2") for n, t in RECIPES19}

print("Rule fixed in advance (PLAN-40): switch the final 12 to 480 epochs + dropout 0.2")
print("  only if the new 12-member set beats the canonical 12 by >= +0.50 points AND >= 1.5 paired SE")
print("  per-recipe rows are descriptive only (not for selection)\n")

for n, _ in RECIPES19:
    r, w = REF19[n], NEW19[n]
    common = sorted(set(r) & set(w))
    print(f"{n:<7s} seeds present: reference {sorted(r)}, new {sorted(w)}")
    if not common:
        continue
    sr = np.mean([hits19([r[s]]) for s in common], axis=0)
    sw = np.mean([hits19([w[s]]) for s in common], axis=0)
    d, se = paired19(sr, sw)
    print(f"        single mean over seeds {common}: new {sw.mean():.4f} ref {sr.mean():.4f}")
    print(f"        delta {d:+.2f} points, paired SE {se:.2f}")
    if len(common) == 3:
        er, ew = hits19([r[s] for s in common]), hits19([w[s] for s in common])
        d2, se2 = paired19(er, ew)
        print(f"        3-seed ensemble: new {ew.mean():.4f} ref {er.mean():.4f}")
        print(f"        delta {d2:+.2f} points, paired SE {se2:.2f}")

REF_ALL = [REF19[n][s] for n, _ in RECIPES19 for s in SEEDS19 if s in REF19[n]]
NEW_ALL = [NEW19[n][s] for n, _ in RECIPES19 for s in SEEDS19 if s in NEW19[n]]
print(f"\nmembers found: canonical {len(REF_ALL)}, new {len(NEW_ALL)}")

if len(REF_ALL) == 12:
    base_acc = float(hits19(REF_ALL).mean())
    print(f"CANONICAL 12: acc {base_acc:.4f}")
    if abs(base_acc - LOGGED19) > 0.0005:
        print("WARN: differs from the logged 0.5993; the canonical files may have changed")

if len(REF_ALL) == 12 and len(NEW_ALL) == 12:
    pm = CONFIG["expected_params"]["resnet_32_64_128"]
    hb, hn = hits19(REF_ALL), hits19(NEW_ALL)
    d, se = paired19(hb, hn)
    met = d >= 0.50 and d >= 1.5 * se
    print(f"NEW 12 (480 epochs + dropout 0.2): acc {hn.mean():.4f}, params {12 * pm:,}")
    print(f"  paired delta {d:+.2f} points, paired SE {se:.2f}, ratio {d / se if se else float('nan'):+.2f}")
    print(f"  -> {'RULE MET' if met else 'rule not met'}")
    print("\nSIZE CURVE (mean accuracy over all subsets): k, canonical, new")
    for k in (1, 2, 3, 4, 6, 12):
        ca = np.mean([hits19([REF_ALL[i] for i in c]).mean() for c in itertools.combinations(range(12), k)])
        na = np.mean([hits19([NEW_ALL[i] for i in c]).mean() for c in itertools.combinations(range(12), k)])
        print(f"  k={k:>2d}  {ca:.4f}  {na:.4f}")
else:
    print("primary comparison skipped: need all 12 members on both sides")

print("Note: paired SE ignores seed sharing and shared folds; the +0.50 / 1.5 SE rule was set before running.")
print("Cell 19 done")

Rule fixed in advance (PLAN-40): switch the final 12 to 480 epochs + dropout 0.2
  only if the new 12-member set beats the canonical 12 by >= +0.50 points AND >= 1.5 paired SE
  per-recipe rows are descriptive only (not for selection)

none    seeds present: reference [0, 1, 2], new [0, 1, 2]
        single mean over seeds [0, 1, 2]: new 0.5867 ref 0.5432
        delta +4.35 points, paired SE 0.63
        3-seed ensemble: new 0.6155 ref 0.5791
        delta +3.64 points, paired SE 0.84
mixup   seeds present: reference [0, 1, 2], new [0, 1, 2]
        single mean over seeds [0, 1, 2]: new 0.5853 ref 0.5496
        delta +3.57 points, paired SE 0.62
        3-seed ensemble: new 0.6094 ref 0.5745
        delta +3.49 points, paired SE 0.86
cutout  seeds present: reference [0, 1, 2], new [0, 1, 2]
        single mean over seeds [0, 1, 2]: new 0.5562 ref 0.5370
        delta +1.92 points, paired SE 0.63
        3-seed ensemble: new 0.5973 ref 0.5730
        delta +2.43 points, paired SE 0.82

In [19]:
# Cell 12: Final training of the 12 ensemble members on all training images
if "apply_experiment" not in globals():
    raise RuntimeError("Run Cells 1-10 in this kernel first (Cell 10 only SKIPs finished runs; it defines apply_experiment)")

FINAL_NAMES = ["DROP_EP480", "DROP_EP480_MIX", "DROP_EP480_CUTOUT", "DROP_EP480_CUTMIX"]
FINAL_DIR = Path(CONFIG["paths"]["results_dir"]) / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_PARAMS = CONFIG["expected_params"]["resnet_32_64_128"]

missing = [n for n in FINAL_NAMES if n not in CONFIG["experiments"]]
if missing:
    raise KeyError(f"Experiments missing from CONFIG: {missing}")


def make_ckpt(state, name, seed, spec, mean, std, params, loss, secs):
    return {
        "state_dict": state,
        "model_key": spec["model"],
        "member": name,
        "seed": seed,
        "mean": mean,
        "std": std,
        "params": params,
        "config": {"spec": dict(spec), "aug": dict(AUG), "train": dict(T)},
        "final_train_loss": float(loss),
        "seconds": float(secs),
        "train_images_sha256": str(DATA_RECORD["train_images_sha256"]),
        "torch_version": str(torch.__version__),
    }


def cpu_flip_logits(model, x, bs=256):
    outs = []
    with torch.no_grad():
        for i in range(0, len(x), bs):
            xb = x[i:i + bs]
            outs.append((model(xb) + model(xb.flip(3))) / 2)
    return torch.cat(outs)


all_idx = np.arange(len(Y_TRAIN))
plan, t_start = [], time.time()

for name in FINAL_NAMES:
    exp = CONFIG["experiments"][name]
    apply_experiment(name)
    spec = {**CONFIG["defaults"], **{k: v for k, v in exp.items() if k in CONFIG["defaults"]}}
    if spec["model"] != "resnet_32_64_128" or spec["epochs"] != 480 or float(T["dropout"]) != 0.2 or not T["flip_avg"]:
        raise ValueError(f"{name}: unexpected final settings (model {spec['model']}, epochs {spec['epochs']}, dropout {T['dropout']}, flip_avg {T['flip_avg']})")

    pre = FINAL_DIR / "_preflight.tmp"
    torch.save(make_ckpt(MODEL_BUILDERS[spec["model"]](NC).state_dict(), name, 0, spec, torch.zeros(3), torch.ones(3), EXPECTED_PARAMS, 0.0, 0.0), pre)
    torch.load(pre, map_location="cpu", weights_only=True)
    pre.unlink()

    for seed in spec["seeds"]:
        path = FINAL_DIR / f"final_{name}_s{seed}.pt"
        plan.append((name, seed, path))
        if path.exists():
            tqdm.write(f"SKIP {path.name}: exists (delete it deliberately to retrain)")
            continue

        out = train_one(spec["model"], spec["lr"], spec["wd"], spec["epochs"], spec["batch_size"], seed, all_idx, None, prefix=f"[final {name} s{seed}] ")
        model = out["model"]
        if out["params"] != EXPECTED_PARAMS:
            raise RuntimeError(f"{name} s{seed}: params {out['params']} != {EXPECTED_PARAMS}")

        tmp = path.with_name(path.name + ".tmp")
        torch.save(make_ckpt({k: v.detach().cpu() for k, v in model.state_dict().items()}, name, seed, spec,
                             out["mean"].cpu(), out["std"].cpu(), out["params"], out["history"][-1][1], out["seconds"]), tmp)
        os.replace(tmp, path)

        loaded = torch.load(path, map_location="cpu", weights_only=True)
        cpu_model = MODEL_BUILDERS[loaded["model_key"]](NC)
        cpu_model.load_state_dict(loaded["state_dict"])
        cpu_model.eval()
        x_cpu = (X_ALL.cpu().float() / 255.0 - loaded["mean"][None, :, None, None]) / loaded["std"][None, :, None, None]
        cpu_logits = cpu_flip_logits(cpu_model, x_cpu)
        gpu_logits = predict_logits(model, normalize(all_idx, out["mean"], out["std"]), 256).cpu()
        mismatch = float((cpu_logits.argmax(dim=1) != gpu_logits.argmax(dim=1)).float().mean())
        max_diff = float((cpu_logits - gpu_logits).abs().max())
        tqdm.write(f"saved {path.name}: params {out['params']}, final train loss {out['history'][-1][1]:.4f}, "
                   f"{out['seconds']:.0f} s | CPU vs GPU (flip-averaged) mismatch {mismatch:.5f}, max |logit diff| {max_diff:.2e}")
        if mismatch > CONFIG["final_check"]["parity_max_mismatch_frac"]:
            raise RuntimeError(f"{path.name}: CPU/GPU mismatch {mismatch:.5f} exceeds the configured limit")
        del out, model, cpu_model, loaded

if len(plan) != 12:
    raise RuntimeError(f"expected 12 members, plan has {len(plan)}")

manifest = []
ref_keys = list(MODEL_BUILDERS["resnet_32_64_128"](NC).state_dict().keys())
for name, seed, path in plan:
    ck = torch.load(path, map_location="cpu", weights_only=True)
    if ck["params"] != EXPECTED_PARAMS or list(ck["state_dict"].keys()) != ref_keys:
        raise RuntimeError(f"{path.name}: params or state_dict keys differ from a fresh resnet_32_64_128")
    if ck["train_images_sha256"] != str(DATA_RECORD["train_images_sha256"]):
        raise RuntimeError(f"{path.name}: trained on different training data")
    manifest.append({"file": path.name, "member": name, "seed": seed, "params": ck["params"],
                     "final_train_loss": ck["final_train_loss"], "seconds": ck["seconds"], "torch_version": ck["torch_version"]})
(FINAL_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"12 members present, total params {sum(m['params'] for m in manifest):,}; wall time this run {time.time() - t_start:.0f} s")
print("Cell 12 done")

[final DROP_EP480 s0]:   0%|          | 0/480 [00:00<?, ?it/s]

[final DROP_EP480 s0] ep  10/480 train_loss 1.9498 val_loss nan val_acc nan lr 0.09989
[final DROP_EP480 s0] ep  20/480 train_loss 1.8362 val_loss nan val_acc nan lr 0.09957
[final DROP_EP480 s0] ep  30/480 train_loss 1.7138 val_loss nan val_acc nan lr 0.09904
[final DROP_EP480 s0] ep  40/480 train_loss 1.6098 val_loss nan val_acc nan lr 0.09830
[final DROP_EP480 s0] ep  50/480 train_loss 1.5199 val_loss nan val_acc nan lr 0.09735
[final DROP_EP480 s0] ep  60/480 train_loss 1.4212 val_loss nan val_acc nan lr 0.09619
[final DROP_EP480 s0] ep  70/480 train_loss 1.2841 val_loss nan val_acc nan lr 0.09484
[final DROP_EP480 s0] ep  80/480 train_loss 1.1968 val_loss nan val_acc nan lr 0.09330
[final DROP_EP480 s0] ep  90/480 train_loss 1.1857 val_loss nan val_acc nan lr 0.09157
[final DROP_EP480 s0] ep 100/480 train_loss 1.1942 val_loss nan val_acc nan lr 0.08967
[final DROP_EP480 s0] ep 110/480 train_loss 1.1330 val_loss nan val_acc nan lr 0.08759
[final DROP_EP480 s0] ep 120/480 train_loss

[final DROP_EP480 s1]:   0%|          | 0/480 [00:00<?, ?it/s]

[final DROP_EP480 s1] ep  10/480 train_loss 1.9366 val_loss nan val_acc nan lr 0.09989
[final DROP_EP480 s1] ep  20/480 train_loss 1.7801 val_loss nan val_acc nan lr 0.09957
[final DROP_EP480 s1] ep  30/480 train_loss 1.6864 val_loss nan val_acc nan lr 0.09904
[final DROP_EP480 s1] ep  40/480 train_loss 1.5739 val_loss nan val_acc nan lr 0.09830
[final DROP_EP480 s1] ep  50/480 train_loss 1.4796 val_loss nan val_acc nan lr 0.09735
[final DROP_EP480 s1] ep  60/480 train_loss 1.3432 val_loss nan val_acc nan lr 0.09619
[final DROP_EP480 s1] ep  70/480 train_loss 1.2493 val_loss nan val_acc nan lr 0.09484
[final DROP_EP480 s1] ep  80/480 train_loss 1.2309 val_loss nan val_acc nan lr 0.09330
[final DROP_EP480 s1] ep  90/480 train_loss 1.1161 val_loss nan val_acc nan lr 0.09157
[final DROP_EP480 s1] ep 100/480 train_loss 1.1731 val_loss nan val_acc nan lr 0.08967
[final DROP_EP480 s1] ep 110/480 train_loss 1.0582 val_loss nan val_acc nan lr 0.08759
[final DROP_EP480 s1] ep 120/480 train_loss

[final DROP_EP480 s2]:   0%|          | 0/480 [00:00<?, ?it/s]

[final DROP_EP480 s2] ep  10/480 train_loss 1.9927 val_loss nan val_acc nan lr 0.09989
[final DROP_EP480 s2] ep  20/480 train_loss 1.8271 val_loss nan val_acc nan lr 0.09957
[final DROP_EP480 s2] ep  30/480 train_loss 1.7408 val_loss nan val_acc nan lr 0.09904
[final DROP_EP480 s2] ep  40/480 train_loss 1.6348 val_loss nan val_acc nan lr 0.09830
[final DROP_EP480 s2] ep  50/480 train_loss 1.5156 val_loss nan val_acc nan lr 0.09735
[final DROP_EP480 s2] ep  60/480 train_loss 1.4200 val_loss nan val_acc nan lr 0.09619
[final DROP_EP480 s2] ep  70/480 train_loss 1.3233 val_loss nan val_acc nan lr 0.09484
[final DROP_EP480 s2] ep  80/480 train_loss 1.2492 val_loss nan val_acc nan lr 0.09330
[final DROP_EP480 s2] ep  90/480 train_loss 1.2457 val_loss nan val_acc nan lr 0.09157
[final DROP_EP480 s2] ep 100/480 train_loss 1.1732 val_loss nan val_acc nan lr 0.08967
[final DROP_EP480 s2] ep 110/480 train_loss 1.1443 val_loss nan val_acc nan lr 0.08759
[final DROP_EP480 s2] ep 120/480 train_loss

... (14 outputs left out)

[final DROP_EP480_CUTMIX s1]:   0%|          | 0/480 [00:00<?, ?it/s]

[final DROP_EP480_CUTMIX s1] ep  10/480 train_loss 2.1820 val_loss nan val_acc nan lr 0.09989
[final DROP_EP480_CUTMIX s1] ep  20/480 train_loss 2.1158 val_loss nan val_acc nan lr 0.09957
[final DROP_EP480_CUTMIX s1] ep  30/480 train_loss 2.1044 val_loss nan val_acc nan lr 0.09904
[final DROP_EP480_CUTMIX s1] ep  40/480 train_loss 2.1249 val_loss nan val_acc nan lr 0.09830
[final DROP_EP480_CUTMIX s1] ep  50/480 train_loss 2.1046 val_loss nan val_acc nan lr 0.09735
[final DROP_EP480_CUTMIX s1] ep  60/480 train_loss 2.1097 val_loss nan val_acc nan lr 0.09619
[final DROP_EP480_CUTMIX s1] ep  70/480 train_loss 2.0711 val_loss nan val_acc nan lr 0.09484
[final DROP_EP480_CUTMIX s1] ep  80/480 train_loss 2.0686 val_loss nan val_acc nan lr 0.09330
[final DROP_EP480_CUTMIX s1] ep  90/480 train_loss 2.0611 val_loss nan val_acc nan lr 0.09157
[final DROP_EP480_CUTMIX s1] ep 100/480 train_loss 2.0620 val_loss nan val_acc nan lr 0.08967
[final DROP_EP480_CUTMIX s1] ep 110/480 train_loss 2.0638 va

[final DROP_EP480_CUTMIX s2]:   0%|          | 0/480 [00:00<?, ?it/s]

[final DROP_EP480_CUTMIX s2] ep  10/480 train_loss 2.2021 val_loss nan val_acc nan lr 0.09989
[final DROP_EP480_CUTMIX s2] ep  20/480 train_loss 2.1644 val_loss nan val_acc nan lr 0.09957
[final DROP_EP480_CUTMIX s2] ep  30/480 train_loss 2.1405 val_loss nan val_acc nan lr 0.09904
[final DROP_EP480_CUTMIX s2] ep  40/480 train_loss 2.1439 val_loss nan val_acc nan lr 0.09830
[final DROP_EP480_CUTMIX s2] ep  50/480 train_loss 2.0947 val_loss nan val_acc nan lr 0.09735
[final DROP_EP480_CUTMIX s2] ep  60/480 train_loss 2.1173 val_loss nan val_acc nan lr 0.09619
[final DROP_EP480_CUTMIX s2] ep  70/480 train_loss 2.0826 val_loss nan val_acc nan lr 0.09484
[final DROP_EP480_CUTMIX s2] ep  80/480 train_loss 2.0520 val_loss nan val_acc nan lr 0.09330
[final DROP_EP480_CUTMIX s2] ep  90/480 train_loss 2.0874 val_loss nan val_acc nan lr 0.09157
[final DROP_EP480_CUTMIX s2] ep 100/480 train_loss 2.0257 val_loss nan val_acc nan lr 0.08967
[final DROP_EP480_CUTMIX s2] ep 110/480 train_loss 2.0399 va